# 🌴 Palm BYOL-YOLO
### Self-Supervised Object Detection for Palm Health Monitoring

*Bootstrapping a YOLOv12s detector with BYOL self-supervised pretraining — squeezing more signal out of labeled data by learning from the unlabeled training images first.*

![Python](https://img.shields.io/badge/Python-3.12-3776AB?logo=python&logoColor=white)
![PyTorch](https://img.shields.io/badge/PyTorch-2.10-EE4C2C?logo=pytorch&logoColor=white)
![Ultralytics](https://img.shields.io/badge/Ultralytics-8.4.72-00FFFF?logo=yolo&logoColor=black)
![SSL](https://img.shields.io/badge/Self--Supervised-BYOL-8A2BE2)
![Domain](https://img.shields.io/badge/Domain-Precision%20Agriculture-2E8B57)
![GPU](https://img.shields.io/badge/GPU-Tesla%20T4-76B900?logo=nvidia&logoColor=white)
![Status](https://img.shields.io/badge/Status-Research%20%2F%20In%20Progress-yellow)

---

## 📚 Table of Contents

- ✨ At a Glance
- 💡 The Idea — Why BYOL?
- 🔁 Pipeline Overview
- 🗂️ Dataset Deep-Dive
- 📑 Stage-by-Stage Notebook Map
- 🧬 BYOL Pretraining — Under the Hood
- 🎯 Detection Fine-Tuning — Under the Hood
- 🔬 Evaluation Suite
- 🛡️ Engineering Safeguards & Design Decisions
- 📦 Output Artifacts
- 🧰 Tech Stack
- 📊 Results
- ▶️ Reproducing This Notebook
- 🙏 Acknowledgments

---

## ✨ At a Glance

| | |
|---|---|
| 🎯 **Task** | 3-class object detection — `abnormal_palm`, `dead_palm`, `healthy_palm` |
| 🧠 **Backbone** | YOLOv12s, pretrained via **BYOL** (Bootstrap Your Own Latent) |
| 🗂️ **Dataset** | Roboflow `palm-oil-vj5ii / dat-palm-fx-vfhm6` — 3,855 train · 200 val · 747 test images |
| 📦 **Annotations** | 9,819 train boxes · 608 val boxes · 2,366 test boxes |
| 🖼️ **SSL Image Size** | 160 × 160 (BYOL views) |
| 🖼️ **Detection Image Size** | 640 × 640 (fine-tuning & eval) |
| ⚙️ **Hardware** | Single Tesla T4 (14.9 GB), Kaggle environment |
| 🧪 **Notebook Structure** | 19 logical stages (39 raw cells incl. markdown headers) |
| 🔁 **Reproducibility** | Global seed 42, deterministic cuDNN, leakage-audited splits |

---

## 💡 The Idea — Why BYOL?

Labeled palm-health imagery is expensive to collect at scale — every box requires an expert to look at a tree and decide *healthy*, *abnormal*, or *dead*. The raw pixels, on the other hand, are cheap.

> **BYOL (Bootstrap Your Own Latent)** learns useful visual representations *without labels* by training a network to predict the representation of one augmented view of an image from a different augmented view of the *same* image — using two networks (an "online" network being trained, and a slowly-moving "target" network) instead of negative pairs or large batches like contrastive methods (SimCLR, MoCo) require.

The hypothesis this notebook tests: if the YOLOv12s backbone/neck is **pretrained with BYOL on the training images alone**, then **fine-tuned normally with labels**, the detector should learn more robust, less overfit features than training from a generic pretrained checkpoint or from scratch — without ever letting validation/test data leak into the self-supervised stage.

---

## 🔁 Pipeline Overview

```
 📥 DATA              🔍 SANITY            🧬 SELF-SUPERVISED        🎯 SUPERVISED          📊 EVALUATION
 ───────────          ───────────          ───────────────────       ──────────────         ─────────────
 Roboflow download      Visual checks        BYOL pretraining          Fine-tune detector      Official mAP
 COCO → YOLO        →   on raw + aug    →    160px · 70 epochs    →   640px · 70 epochs   →   Matched-event stats
 + offline augment      Class distros        EMA target, m=0.996      AdamW + cosine LR        Calibration & cost
                                              backbone/neck only       shape-safe weight        Threshold robustness
                                                                       transfer                 Paper-ready tables
```

Each phase is leakage-isolated: augmentation, BYOL pretraining, and fine-tuning all read exclusively from the **training** split; validation and test images are touched only at evaluation time, and only in their **original, unaugmented** form.

---

## 🗂️ Dataset Deep-Dive

**Source:** Roboflow project `palm-oil-vj5ii / dat-palm-fx-vfhm6`, version 1, COCO format.

### Conversion & Integrity Audit (Cell 3)

| Split | Images (JSON) | Annotations (JSON) | Copied | Valid Boxes | Invalid/Skipped Boxes | Images w/o Valid Labels |
|---|---:|---:|---:|---:|---:|---:|
| Train | 3,855 | 9,819 | 3,855 | 9,819 | 0 | 0 |
| Valid | 200 | 608 | 200 | 608 | 0 | 0 |
| Test | 747 | 2,366 | 747 | 2,366 | 0 | 0 |

> **🛡️ Leakage audit:** MD5 image-hash comparison across all three splits found **zero duplicate images** between train/valid/test.

### Class Mapping

Category IDs are derived **from the training annotations only**. A 4th metadata category (`palms-obj`) appears in the raw COCO file but has **zero training instances**, so it is explicitly excluded from the class list rather than silently included as an empty class.

| Final Class ID | Class Name |
|---:|---|
| 0 | `abnormal_palm` |
| 1 | `dead_palm` |
| 2 | `healthy_palm` |

### Per-Class Instance Distribution

| Class | Train | Valid | Test |
|---|---:|---:|---:|
| `abnormal_palm` | 2,679 | 128 | 602 |
| `dead_palm` | 2,152 | 64 | 245 |
| `healthy_palm` | 4,988 | 416 | 1,519 |
| **Total** | **9,819** | **608** | **2,366** |

### Offline Training Augmentation (Cell 5)

Generates up to **1,000** extra training images (3 attempts per source image), using Albumentations with `BboxParams(format="yolo", min_visibility=0.25, min_area=4)`:

**Geometric**
- `HorizontalFlip(p=0.5)`
- `ShiftScaleRotate(shift_limit=0.06, scale_limit=0.15, rotate_limit=12°, border_mode=REFLECT_101, p=0.60)`
- *(optional, disabled by default)* `VerticalFlip` / `RandomRotate90` — gated behind `USE_STRONG_ORIENTATION_AUG=False` since the imagery isn't confirmed nadir/top-down

**Photometric**
- `OneOf[RandomBrightnessContrast(±0.18), CLAHE(clip=2.0, tile=8×8)], p=0.55`
- `HueSaturationValue(hue=8, sat=25, val=18, p=0.35)`
- `OneOf[GaussianBlur(3–5), GaussNoise(var=5–20)], p=0.18`

Validation and test splits are copied through **unaugmented**.

---

## 📑 Stage-by-Stage Notebook Map

| # | Stage | What it does |
|---|---|---|
| 1 | 🛠️ Environment Setup | Installs deps, seeds Python/NumPy/Torch, deterministic cuDNN, verifies CUDA/GPU |
| 2 | 📥 Roboflow Download | Pulls the COCO-format dataset (skips re-download if already cached) |
| 3 | 🔄 COCO → YOLO Conversion | Train-derived category mapping, bbox clipping/validation, MD5 leakage audit |
| 4 | 👁️ Annotation Sanity Check | Draws boxes on sampled training images, flags malformed label lines |
| 5 | 🎨 Training-Only Augmentation | Geometric + photometric Albumentations pipeline, train split only |
| 6 | 📈 Distribution Audit | Before/after image & per-class instance counts, bar chart |
| 7 | 👁️ Augmented Sample Check | Visual QA pass on the augmented training set |
| 8 | ⚙️ BYOL + YOLOv12s Config | Canonical paths, reproducibility, required-file validation |
| 9 | 🧩 BYOL Dataset | `TwoView` two-augmentation dataset, train-only leakage guard, image-integrity check |
| 10 | 🧠 BYOL Loss & Hooks | Negative-cosine loss, Detect-head feature hooks, EMA + momentum schedule, MLP builders |
| 11 | 🚀 BYOL Pretraining | 70-epoch SSL run on backbone/neck only, early stopping, best-checkpoint saving |
| 12 | 🎯 Detection Fine-Tuning | Shape-safe BYOL weight transfer, full-detector training |
| 13 | ✅ Official Val/Test Eval | Native Ultralytics `.val()` on the *original* (non-augmented) split |
| 14 | 📉 Training Curves | Loss/metric/LR plots parsed from `results.csv` |
| 15 | 🔬 Matched-Event Evaluation | IoU-matched classification-style diagnostics: κ, ECE, Brier, ROC/PR |
| 16 | ⏱️ Cost Benchmarking | Params, GFLOPs, latency distribution, throughput, peak GPU memory |
| 17 | 🖼️ Qualitative Results | Sample detections rendered on test images |
| 18 | 🎚️ Threshold Robustness | F1 sweep across 9 confidence × 5 IoU thresholds (45 combinations) |
| 19 | 📄 Paper-Ready Tables | Aggregates everything into 6 publication tables (CSV + Markdown + LaTeX) |

---

## 🧬 BYOL Pretraining — Under the Hood

#### Hyperparameters

| Parameter | Value |
|---|---|
| Epochs | 70 |
| Batch size | 8 |
| Image size | 160 × 160 |
| Optimizer | AdamW |
| Learning rate | 1e-3 |
| Weight decay | 1e-4 |
| Base EMA momentum | 0.996 → 1.0 (cosine schedule) |
| Gradient clipping | max-norm 5.0 |
| Mixed precision | AMP (`GradScaler`) |
| Early stopping | patience 10, min Δ 1e-4 |
| Checkpoint scope | backbone + neck only (Detect head excluded) |

**View augmentation** — deliberately gentler than typical natural-image SSL:

```
RandomResizedCrop(160, scale=(0.45, 1.0))   # vs. the SSL-standard (0.20, 1.0)
RandomHorizontalFlip(p=0.5)
ColorJitter(brightness=0.35, contrast=0.35, saturation=0.30, hue=0.08), p=0.75
RandomGrayscale(p=0.15)
GaussianBlur(kernel=3, sigma=0.1–1.5), p=0.20
Normalize(ImageNet mean/std)
```

> **💡 Design rationale:** the standard SSL crop range `(0.2, 1.0)` is tuned for natural-image classification, where the subject usually fills the frame. For small-object detection, an aggressive crop can remove the object entirely from both views — so the lower bound is raised to `0.45`.

**Architecture**

- **Feature extraction:** a forward pre-hook on the YOLO `Detect` module captures the P3/P4/P5 multi-scale feature maps *before* they reach the detect head, on both the online and target networks. The target's Detect head is forced into train-mode (while the rest of the target stays in eval-mode) specifically to bypass YOLO's inference-time box-decoding path and expose raw feature maps instead.
- **Projector / Predictor:** `Linear → BatchNorm → ReLU → Linear`, mapping pooled features → 512 hidden → 256 output. The target projector is a frozen copy of the online projector (no predictor on the target side — standard BYOL asymmetry).
- **Loss:**
  ```
  L = 2 − 2 · cos_sim(p, stopgrad(z))
  Total = L(p1, z2_target) + L(p2, z1_target)     # symmetrized over both views
  ```
- **EMA update order:** the target network is updated with EMA *after* the optimizer step on the online network — explicitly called out in-notebook since getting this ordering backwards is a common BYOL implementation bug.

---

## 🎯 Detection Fine-Tuning — Under the Hood

#### Hyperparameters

| Parameter | Value |
|---|---|
| Epochs | 70 |
| Image size | 640 × 640 |
| Batch size | 8 |
| Optimizer | AdamW |
| `lr0` / `lrf` | 5e-4 / 0.01 (cosine schedule) |
| Weight decay | 5e-4 |
| Warmup epochs | 3 |
| Early stopping patience | 10 |
| Mosaic | 0.40 (closes at epoch 10) |
| Mixup / Copy-paste / Erasing | 0 (disabled) |
| HSV jitter (h/s/v) | 0.010 / 0.400 / 0.250 |
| Geometric jitter | degrees 5°, translate 0.05, scale 0.25, fliplr 0.50 |

> **💡 Design rationale — no double augmentation:** because offline augmentation already ran in Cell 5, YOLO's *online* augmentation is deliberately turned down here (mixup/copy-paste/erasing off, modest mosaic/HSV/geometric jitter) to avoid compounding two augmentation pipelines on top of each other.

**Weight transfer.** `load_byol_weights_safely` filters the BYOL checkpoint by **both** key existence *and* tensor-shape match before calling `load_state_dict` — stricter than plain `strict=False`, which silently skips key mismatches but does *not* catch shape mismatches on matching keys. Compatible / missing / unexpected / shape-mismatched keys are all reported, and the cell raises if zero tensors transferred successfully.

**Model resolution.** Both the BYOL cell and the fine-tuning cell try `yolo12s.pt → yolov12s.pt → yolo12s.yaml → yolov12s.yaml` in order, defensively, since Ultralytics' naming for YOLOv12 has shifted across package versions.

---

## 🧰 Tech Stack

`Ultralytics YOLOv12` · `PyTorch` · `Albumentations` · `Roboflow` · `scikit-learn` · `OpenCV` · `Matplotlib` · `Pandas` · `NumPy`

---

## 🙏 Acknowledgments

Dataset hosted via **Roboflow** (`palm-oil-vj5ii / dat-palm-fx-vfhm6`). Detector built on **Ultralytics YOLOv12**. Self-supervised pretraining follows the **BYOL** method (Grill et al., 2020), adapted here for detection backbones rather than image classifiers.

*Made with 🧠 self-supervised learning and a lot of leakage audits.*

## Cell 1 — Environment Setup, Reproducibility, Imports

In [ ]:
# ============================================================
# Cell 1 — Environment Setup, Reproducibility, Imports
# ============================================================

import os
import gc
import cv2
import json
import yaml
import math
import time
import random
import shutil
import hashlib
import contextlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# Kaggle input inspection
print("Kaggle input files:")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Install required packages
%pip -q install ultralytics timm supervision roboflow albumentations psutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
from ultralytics import YOLO
import ultralytics

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility configuration
# -----------------------------
SEED = 42

def seed_everything(seed: int = 42):
    """
    Sets random seeds for reproducible experiments.
    Full bit-level reproducibility is not always guaranteed on GPU,
    but this substantially reduces uncontrolled randomness.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

seed_everything(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("=" * 70)
print("Environment Check")
print("=" * 70)
print(f"Device       : {device}")
print(f"PyTorch      : {torch.__version__}")
print(f"Ultralytics  : {ultralytics.__version__}")
if device == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version : {torch.version.cuda}")

ultralytics.checks()

def autocast_ctx():
    """
    Mixed precision context.
    Keeps CPU execution safe by returning a null context when CUDA is unavailable.
    """
    if device == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    return contextlib.nullcontext()

## Cell 2 — Roboflow Dataset Download

In [ ]:
# ============================================================
# Cell 2 — Roboflow Dataset Download
# ============================================================

from pathlib import Path
import os
from roboflow import Roboflow

BASE = Path("/kaggle/working")

# Expected Roboflow dataset directory
DATASET_DIR = BASE / "Dat-Palm-Fx-1"

# Project workspace
WORK  = BASE / "palm_byol_ssl"
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_byol.yaml"

# SSL backbone weights
SSL_W = WORK / "byol_backbone_with_hooks_yolov12.pt"

# Keep original variable name to avoid breaking later cells
YOL0V12S_W = WORK / "yolov12s.pt"

# Optional safer alias for readability in future cells
YOLOV12S_W = YOL0V12S_W

WORK.mkdir(parents=True, exist_ok=True)
SPLIT.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("Project Paths")
print("=" * 70)
print("BASE        :", BASE)
print("DATASET_DIR :", DATASET_DIR)
print("WORK        :", WORK)
print("SPLIT       :", SPLIT)
print("DATA        :", DATA)
print("SSL_W       :", SSL_W)
print("YOL0V12S_W  :", YOL0V12S_W)


# ------------------------------------------------------------
# Roboflow API key
# ------------------------------------------------------------
# For final public/journal version, move this key to Kaggle Secrets.
rf_key = "kb3t7HveaF3ISN0TEWpk"


required_ann = DATASET_DIR / "train" / "_annotations.coco.json"

if required_ann.exists():
    print(f"Dataset already exists at: {DATASET_DIR}")

else:
    rf = Roboflow(api_key=rf_key)
    project = rf.workspace("palm-oil-vj5ii").project("dat-palm-fx-vfhm6")
    version = project.version(1)
    dataset = version.download("coco")

    DATASET_DIR = Path(dataset.location)
    print("Downloaded dataset to:", DATASET_DIR)


# Final dataset sanity check
for split_name in ["train", "valid", "test"]:
    ann_path = DATASET_DIR / split_name / "_annotations.coco.json"
    img_dir = DATASET_DIR / split_name

    if not ann_path.exists():
        raise FileNotFoundError(f"Missing COCO annotation file: {ann_path}")

    if not img_dir.exists():
        raise FileNotFoundError(f"Missing image directory: {img_dir}")

print("=" * 70)
print("Dataset path verified")
print("=" * 70)
print("DATASET_DIR:", DATASET_DIR)

## Cell 3 — COCO to YOLO Conversion

In [ ]:
# ============================================================
# Cell 3 — COCO to YOLO Conversion with Split Integrity Audit
# ============================================================

BASE        = Path("/kaggle/working")
WORK        = BASE / "palm_byol_ssl"
SPLIT       = WORK / "0_yolo_split"
DATA        = WORK / "data_byol.yaml"

SSL_W       = WORK / "byol_backbone_with_hooks_yolov12.pt"
YOL0V12S_W  = WORK / "yolov12s.pt"
YOLOV12S_W  = YOL0V12S_W

WORK.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)


def coco_bbox_to_yolo_safe(bbox, img_w, img_h):
    """
    Converts COCO bbox [x, y, width, height] to normalized YOLO format.
    The bbox is clipped to image boundaries.

    Returns:
        tuple or None
    """
    x, y, bw, bh = map(float, bbox)

    if img_w <= 0 or img_h <= 0 or bw <= 0 or bh <= 0:
        return None

    x1 = max(0.0, x)
    y1 = max(0.0, y)
    x2 = min(float(img_w), x + bw)
    y2 = min(float(img_h), y + bh)

    clipped_w = x2 - x1
    clipped_h = y2 - y1

    if clipped_w <= 1e-6 or clipped_h <= 1e-6:
        return None

    x_center = (x1 + clipped_w / 2.0) / img_w
    y_center = (y1 + clipped_h / 2.0) / img_h
    norm_w   = clipped_w / img_w
    norm_h   = clipped_h / img_h

    values = [x_center, y_center, norm_w, norm_h]

    if any((v < 0 or v > 1 or not np.isfinite(v)) for v in values):
        return None

    return tuple(values)


def file_hash(path: Path, block_size: int = 65536):
    """
    Computes image file hash to detect exact duplicates across splits.
    """
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()


def image_integrity_check(path: Path):
    """
    Checks whether an image can be opened and has valid dimensions.
    """
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            w, h = img.size

        if w <= 0 or h <= 0:
            return False, w, h

        return True, w, h

    except Exception:
        return False, np.nan, np.nan


def build_train_category_mapping(train_ann_path: Path):
    """
    Builds category mapping using training annotations only.

    This avoids allowing validation/test-only categories to influence
    the training label space. Categories with zero training instances
    are excluded and reported.
    """
    train_coco = load_json(train_ann_path)

    all_categories = sorted(train_coco.get("categories", []), key=lambda x: x["id"])
    train_present_cat_ids = set(a["category_id"] for a in train_coco.get("annotations", []))

    categories = [c for c in all_categories if c["id"] in train_present_cat_ids]
    empty_cats = [c for c in all_categories if c["id"] not in train_present_cat_ids]

    if len(categories) == 0:
        raise ValueError("No valid categories found in training annotations.")

    id2new = {cat["id"]: i for i, cat in enumerate(categories)}
    names = [cat["name"] for cat in categories]

    print("=" * 70)
    print("Category Mapping")
    print("=" * 70)

    if empty_cats:
        print("Categories present in metadata but absent from training annotations:")
        for c in empty_cats:
            print(f"  excluded: original_id={c['id']} | name={c['name']}")

    print("\nUsing train-derived category mapping:")
    for orig_id, new_id in id2new.items():
        cat_name = next(c["name"] for c in categories if c["id"] == orig_id)
        print(f"  original_id={orig_id} -> yolo_id={new_id} | name={cat_name}")

    return id2new, names, categories


def convert_coco_split_to_yolo(split_name, img_dir, ann_json_path, id2new):
    """
    Converts one COCO split to YOLO format.

    Key safeguards:
    - creates empty label files for images without valid boxes
    - clips invalid bboxes safely
    - skips categories absent from train-derived mapping
    - reports invalid/missing/corrupt samples
    """
    img_dir = Path(img_dir)
    ann_json_path = Path(ann_json_path)

    out_img_dir = SPLIT / split_name / "images"
    out_lbl_dir = SPLIT / split_name / "labels"

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    coco = load_json(ann_json_path)

    images = coco.get("images", [])
    annotations = coco.get("annotations", [])

    id2img = {
        im["id"]: {
            "id": im["id"],
            "file_name": im["file_name"],
            "width": im.get("width", None),
            "height": im.get("height", None),
        }
        for im in images
    }

    image_to_annotations = defaultdict(list)
    for ann in annotations:
        image_to_annotations[ann["image_id"]].append(ann)

    audit = {
        "split": split_name,
        "num_images_in_json": len(images),
        "num_annotations_in_json": len(annotations),
        "copied_images": 0,
        "missing_images": 0,
        "corrupt_images": 0,
        "valid_boxes": 0,
        "invalid_boxes": 0,
        "skipped_category_boxes": 0,
        "images_without_valid_labels": 0,
    }

    class_counter = Counter()

    for im in images:
        file_name = im["file_name"]
        src_img = img_dir / file_name
        dst_img = out_img_dir / file_name
        lbl_path = out_lbl_dir / f"{Path(file_name).stem}.txt"

        label_lines = []

        if not src_img.exists():
            audit["missing_images"] += 1
            lbl_path.write_text("")
            continue

        ok, actual_w, actual_h = image_integrity_check(src_img)
        if not ok:
            audit["corrupt_images"] += 1
            lbl_path.write_text("")
            continue

        # Prefer actual dimensions over COCO metadata if available.
        img_w = int(actual_w)
        img_h = int(actual_h)

        anns = image_to_annotations.get(im["id"], [])

        for ann in anns:
            original_cat_id = ann.get("category_id")

            if original_cat_id not in id2new:
                audit["skipped_category_boxes"] += 1
                continue

            yolo_box = coco_bbox_to_yolo_safe(ann.get("bbox", []), img_w, img_h)

            if yolo_box is None:
                audit["invalid_boxes"] += 1
                continue

            cls_id = id2new[original_cat_id]
            class_counter[cls_id] += 1

            label_lines.append(
                f"{cls_id} " + " ".join(f"{v:.6f}" for v in yolo_box)
            )
            audit["valid_boxes"] += 1

        if len(label_lines) == 0:
            audit["images_without_valid_labels"] += 1

        lbl_path.write_text("\n".join(label_lines) + ("\n" if label_lines else ""))

        shutil.copy2(src_img, dst_img)
        audit["copied_images"] += 1

    return audit, class_counter


def audit_split_hash_leakage(split_root: Path):
    """
    Detects exact duplicate image files across train/valid/test.
    Exact duplicate leakage can inflate validation/test results.
    """
    rows = []

    for split_name in ["train", "valid", "test"]:
        image_dir = split_root / split_name / "images"
        for img_path in sorted(image_dir.glob("*")):
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue

            try:
                rows.append({
                    "split": split_name,
                    "file": img_path.name,
                    "path": str(img_path),
                    "md5": file_hash(img_path),
                })
            except Exception:
                rows.append({
                    "split": split_name,
                    "file": img_path.name,
                    "path": str(img_path),
                    "md5": None,
                })

    df_hash = pd.DataFrame(rows)

    leakage_rows = []

    if not df_hash.empty:
        for md5, group in df_hash.dropna(subset=["md5"]).groupby("md5"):
            splits = sorted(group["split"].unique())
            if len(splits) > 1:
                leakage_rows.append({
                    "md5": md5,
                    "splits": ", ".join(splits),
                    "files": "; ".join(group["file"].tolist()),
                })

    df_leakage = pd.DataFrame(leakage_rows)
    return df_hash, df_leakage


def count_yolo_labels(split_root: Path, names):
    """
    Counts YOLO labels per split and per class.
    """
    rows = []

    for split_name in ["train", "valid", "test"]:
        label_dir = split_root / split_name / "labels"
        image_dir = split_root / split_name / "images"

        split_counter = Counter()
        total_boxes = 0

        label_files = sorted(label_dir.glob("*.txt"))
        image_files = sorted([p for p in image_dir.glob("*") if p.suffix.lower() in IMAGE_EXTS])

        for lbl_path in label_files:
            lines = [ln.strip() for ln in lbl_path.read_text().splitlines() if ln.strip()]
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    split_counter[cls_id] += 1
                    total_boxes += 1

        for cls_id, cls_name in enumerate(names):
            rows.append({
                "split": split_name,
                "class_id": cls_id,
                "class_name": cls_name,
                "instances": split_counter.get(cls_id, 0),
                "num_images": len(image_files),
                "num_label_files": len(label_files),
                "total_boxes": total_boxes,
            })

    return pd.DataFrame(rows)


# -----------------------------
# Build split safely
# -----------------------------
train_ann_path = DATASET_DIR / "train" / "_annotations.coco.json"

if not train_ann_path.exists():
    raise FileNotFoundError(f"Train annotation file not found: {train_ann_path}")

id2new, names, categories = build_train_category_mapping(train_ann_path)

if SPLIT.exists():
    shutil.rmtree(SPLIT)
    print("\nOld YOLO split removed:", SPLIT)

print("\n" + "=" * 70)
print("COCO to YOLO Conversion")
print("=" * 70)

split_audits = []
split_class_counters = {}

for split_name in ["train", "valid", "test"]:
    ann_path = DATASET_DIR / split_name / "_annotations.coco.json"
    img_dir = DATASET_DIR / split_name

    audit, class_counter = convert_coco_split_to_yolo(
        split_name=split_name,
        img_dir=img_dir,
        ann_json_path=ann_path,
        id2new=id2new,
    )

    split_audits.append(audit)
    split_class_counters[split_name] = class_counter

df_conversion_audit = pd.DataFrame(split_audits)

print("\nConversion audit:")
display(df_conversion_audit)

# -----------------------------
# Write YOLO data.yaml
# -----------------------------
DATA.parent.mkdir(parents=True, exist_ok=True)

data_yaml = {
    "path": str(SPLIT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(names),
    "names": names,
}

DATA.write_text(yaml.dump(data_yaml, sort_keys=False))

print("\nYOLO data.yaml written to:", DATA)
print(yaml.dump(data_yaml, sort_keys=False))

# -----------------------------
# Dataset distribution audit
# -----------------------------
df_label_distribution = count_yolo_labels(SPLIT, names)

print("=" * 70)
print("Class Distribution After Conversion")
print("=" * 70)
display(df_label_distribution)

# -----------------------------
# Exact duplicate leakage audit
# -----------------------------
print("=" * 70)
print("Exact Image Duplicate Leakage Check")
print("=" * 70)

df_hashes, df_split_leakage = audit_split_hash_leakage(SPLIT)

if df_split_leakage.empty:
    print("No exact duplicate image leakage detected across train/valid/test splits.")
else:
    print("WARNING: Exact duplicate images detected across splits.")
    display(df_split_leakage)

# -----------------------------
# Final summary
# -----------------------------
print("=" * 70)
print("YOLO Split Ready")
print("=" * 70)
print("SPLIT :", SPLIT)
print("DATA  :", DATA)
print("Names :", names)
print("Number of classes:", len(names))

## Cell 4 — Annotation Visualization and Sanity Check

In [ ]:
# ============================================================
# Cell 4 — Annotation Visualization and Sanity Check
# ============================================================

from pathlib import Path
import math
import random
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# -----------------------------
# Paths
# -----------------------------
BASE        = Path("/kaggle/working")
DATASET_DIR = BASE / "Dat-Palm-Fx-1"

WORK  = BASE / "palm_byol_ssl"
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_byol.yaml"

SSL_W      = WORK / "byol_backbone_with_hooks_yolov12.pt"
YOL0V12S_W = WORK / "yolov12s.pt"
YOLOV12S_W = YOL0V12S_W

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# -----------------------------
# Load YOLO data configuration
# -----------------------------
if not DATA.exists():
    raise FileNotFoundError(f"YOLO data YAML not found: {DATA}")

with open(DATA, "r") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg.get("names", None)

if class_names is None or len(class_names) == 0:
    raise ValueError("No class names found in data YAML.")

print("=" * 70)
print("YOLO Dataset Configuration")
print("=" * 70)
print("Classes:", class_names)
print("Number of classes:", len(class_names))


# ============================================================
# Helper functions
# ============================================================

def load_yolo_labels(lbl_path: Path, num_classes: int = None):
    """
    Loads YOLO-format labels safely.

    Expected format:
        class_id x_center y_center width height

    Returns:
        boxes: list of tuples -> (cls, x_c, y_c, w, h)
        issues: list of warning strings
    """
    boxes = []
    issues = []

    lbl_path = Path(lbl_path)

    if not lbl_path.exists():
        issues.append(f"Missing label file: {lbl_path.name}")
        return boxes, issues

    with open(lbl_path, "r") as f:
        for line_idx, line in enumerate(f, start=1):
            line = line.strip()

            if line == "":
                continue

            parts = line.split()

            if len(parts) != 5:
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: expected 5 values, got {len(parts)}"
                )
                continue

            try:
                cls = int(float(parts[0]))
                x_c, y_c, bw, bh = map(float, parts[1:])
            except Exception:
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: non-numeric label values"
                )
                continue

            if num_classes is not None and not (0 <= cls < num_classes):
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: class id {cls} outside valid range"
                )
                continue

            vals = [x_c, y_c, bw, bh]

            if not all(np.isfinite(vals)):
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: non-finite bbox values"
                )
                continue

            if bw <= 0 or bh <= 0:
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: bbox width/height must be positive"
                )
                continue

            # YOLO normalized coordinates should be within [0, 1].
            # Small boundary violations are clipped during drawing but reported here.
            if any(v < 0 or v > 1 for v in vals):
                issues.append(
                    f"{lbl_path.name} | line {line_idx}: bbox values outside [0, 1]"
                )

            boxes.append((cls, x_c, y_c, bw, bh))

    return boxes, issues


def yolo_to_xyxy(box, img_w: int, img_h: int):
    """
    Converts YOLO normalized bbox to clipped pixel xyxy coordinates.
    """
    cls, x_c, y_c, bw, bh = box

    x_c_abs = x_c * img_w
    y_c_abs = y_c * img_h
    bw_abs  = bw * img_w
    bh_abs  = bh * img_h

    x1 = int(round(x_c_abs - bw_abs / 2))
    y1 = int(round(y_c_abs - bh_abs / 2))
    x2 = int(round(x_c_abs + bw_abs / 2))
    y2 = int(round(y_c_abs + bh_abs / 2))

    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))

    if x2 <= x1 or y2 <= y1:
        return None

    return x1, y1, x2, y2


def get_class_colors(num_classes: int):
    """
    Generates stable class colors for annotation visualization.
    """
    rng = np.random.default_rng(SEED)
    colors = rng.integers(low=40, high=230, size=(num_classes, 3), dtype=np.uint8)
    return [tuple(map(int, c)) for c in colors]


CLASS_COLORS = get_class_colors(len(class_names))


def draw_yolo_boxes(img_path: Path, lbl_path: Path, class_names):
    """
    Draws YOLO boxes on an image.

    Returns:
        RGB image with boxes, list of labels, list of issues
    """
    img_path = Path(img_path)
    lbl_path = Path(lbl_path)

    img = cv2.imread(str(img_path))

    if img is None:
        return None, [], [f"Could not read image: {img_path.name}"]

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    boxes, issues = load_yolo_labels(lbl_path, num_classes=len(class_names))
    drawn_boxes = []

    for box in boxes:
        cls, x_c, y_c, bw, bh = box
        xyxy = yolo_to_xyxy(box, img_w, img_h)

        if xyxy is None:
            issues.append(f"{lbl_path.name}: invalid bbox after pixel conversion")
            continue

        x1, y1, x2, y2 = xyxy

        color = CLASS_COLORS[cls]
        cls_name = class_names[cls] if cls < len(class_names) else str(cls)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        label = f"{cls_name}"
        font_scale = 0.45
        thickness = 1

        (text_w, text_h), baseline = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            thickness
        )

        text_x = x1
        text_y = max(text_h + 4, y1 - 4)

        cv2.rectangle(
            img,
            (text_x, text_y - text_h - baseline - 4),
            (text_x + text_w + 4, text_y + baseline),
            color,
            -1
        )

        cv2.putText(
            img,
            label,
            (text_x + 2, text_y - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA
        )

        drawn_boxes.append(cls_name)

    return img, drawn_boxes, issues


def collect_split_images(split_name: str):
    """
    Collects image paths from a YOLO split.
    """
    img_dir = SPLIT / split_name / "images"

    if not img_dir.exists():
        raise FileNotFoundError(f"Image directory not found: {img_dir}")

    img_paths = sorted(
        [p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
    )

    return img_paths


# ============================================================
# Visualize training annotations
# ============================================================

train_img_dir = SPLIT / "train" / "images"
train_lbl_dir = SPLIT / "train" / "labels"

all_imgs = collect_split_images("train")

if len(all_imgs) == 0:
    raise RuntimeError(f"No training images found in: {train_img_dir}")

n_show = min(8, len(all_imgs))

# Reproducible random sample
rng = random.Random(SEED)
sample_imgs = rng.sample(all_imgs, n_show)

cols = 4
rows = math.ceil(n_show / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 4.8 * rows), dpi=160)
axes = np.array(axes).reshape(-1) if rows * cols > 1 else np.array([axes])

all_issues = []
sample_summary = []

for i, ax in enumerate(axes):
    ax.axis("off")

    if i >= n_show:
        continue

    img_path = sample_imgs[i]
    lbl_path = train_lbl_dir / f"{img_path.stem}.txt"

    vis, drawn_labels, issues = draw_yolo_boxes(img_path, lbl_path, class_names)

    all_issues.extend(issues)

    sample_summary.append({
        "image": img_path.name,
        "label_file": lbl_path.name,
        "num_boxes_drawn": len(drawn_labels),
        "classes_present": ", ".join(sorted(set(drawn_labels))) if drawn_labels else "None"
    })

    if vis is None:
        ax.set_title(f"Unreadable image\n{img_path.name}", fontsize=9)
        continue

    ax.imshow(vis)
    ax.set_title(
        f"Sample {i + 1}: {img_path.name}\nBoxes: {len(drawn_labels)}",
        fontsize=9
    )

fig.suptitle(
    "Training Set Annotation Sanity Check: YOLO Bounding Boxes",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.94])

fig_path = FIG_DIR / "training_annotation_sanity_check.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("=" * 70)
print("Annotation Visualization Summary")
print("=" * 70)
print(f"Displayed samples : {n_show}")
print(f"Figure saved to   : {fig_path}")

df_sample_summary = pd.DataFrame(sample_summary)
display(df_sample_summary)

if all_issues:
    print("=" * 70)
    print("Annotation Warnings Found in Displayed Samples")
    print("=" * 70)

    issue_counter = Counter(all_issues)
    df_issues = pd.DataFrame(
        [{"issue": issue, "count": count} for issue, count in issue_counter.items()]
    )
    display(df_issues)
else:
    print("No annotation format issues detected in the displayed samples.")

## Cell 5 — Training-Only Detection Augmentation

In [ ]:
# ============================================================
# Cell 5 — Training-Only Detection Augmentation with Audit
# ============================================================

%pip -q install albumentations==1.4.7

from pathlib import Path
import os
import cv2
import yaml
import shutil
import random
import numpy as np
import pandas as pd
import albumentations as A
from tqdm.auto import tqdm
from collections import Counter, defaultdict

# -----------------------------
# Paths
# -----------------------------
BASE  = Path("/kaggle/working")
WORK  = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_byol.yaml"

# New augmented dataset root
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG  = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Maximum number of additional augmented training images
MAX_AUG_IMAGES = 1000

# Number of augmentation attempts per eligible training image
N_AUG_PER_IMAGE = 3

# Safer default:
# Keep False unless the dataset is confirmed to be top-down/nadir imagery
# where vertical and 90-degree rotations are physically valid.
USE_STRONG_ORIENTATION_AUG = False

print("=" * 70)
print("Augmentation Configuration")
print("=" * 70)
print("Original split :", SPLIT)
print("Augmented split:", AUG_SPLIT)
print("Max augmented training images:", MAX_AUG_IMAGES)
print("Augmentation only applied to: train")
print("Validation/test augmentation: disabled")


# -----------------------------
# Load dataset YAML
# -----------------------------
if not DATA.exists():
    raise FileNotFoundError(f"Original YOLO data YAML not found: {DATA}")

with open(DATA, "r") as f:
    cfg = yaml.safe_load(f)

class_names = cfg.get("names", [])
num_classes = len(class_names)

if num_classes == 0:
    raise ValueError("No class names found in data YAML.")

print("Classes:", class_names)


# ============================================================
# Label utilities
# ============================================================

def sanitize_box(xc, yc, w, h, eps=1e-6):
    """
    Clamp a normalized YOLO bbox so that its corners stay inside [0, 1].

    Returns:
        (xc, yc, w, h) or None if degenerate/invalid.
    """
    values = [xc, yc, w, h]

    if not all(np.isfinite(values)):
        return None

    if w <= 0 or h <= 0:
        return None

    x1 = xc - w / 2.0
    y1 = yc - h / 2.0
    x2 = xc + w / 2.0
    y2 = yc + h / 2.0

    x1 = max(0.0, min(1.0, x1))
    y1 = max(0.0, min(1.0, y1))
    x2 = max(0.0, min(1.0, x2))
    y2 = max(0.0, min(1.0, y2))

    new_w = x2 - x1
    new_h = y2 - y1

    if new_w < eps or new_h < eps:
        return None

    new_xc = (x1 + x2) / 2.0
    new_yc = (y1 + y2) / 2.0

    return new_xc, new_yc, new_w, new_h


def load_yolo_labels_safe(lbl_path: Path, num_classes: int):
    """
    Loads and sanitizes YOLO labels.

    Returns:
        boxes: list of (cls, xc, yc, w, h)
        stats: dictionary with audit information
    """
    boxes = []
    stats = {
        "missing_label_file": 0,
        "invalid_label_lines": 0,
        "invalid_class_ids": 0,
        "invalid_boxes": 0,
        "valid_boxes": 0,
    }

    lbl_path = Path(lbl_path)

    if not lbl_path.exists():
        stats["missing_label_file"] = 1
        return boxes, stats

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) == 0:
                continue

            if len(parts) != 5:
                stats["invalid_label_lines"] += 1
                continue

            try:
                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:])
            except Exception:
                stats["invalid_label_lines"] += 1
                continue

            if not (0 <= cls < num_classes):
                stats["invalid_class_ids"] += 1
                continue

            sanitized = sanitize_box(xc, yc, bw, bh)

            if sanitized is None:
                stats["invalid_boxes"] += 1
                continue

            xc, yc, bw, bh = sanitized
            boxes.append((cls, xc, yc, bw, bh))
            stats["valid_boxes"] += 1

    return boxes, stats


def save_yolo_labels(lbl_path: Path, boxes):
    """
    Saves YOLO labels. Creates an empty file when boxes are empty.
    """
    lbl_path.parent.mkdir(parents=True, exist_ok=True)

    with open(lbl_path, "w") as f:
        for cls, xc, yc, bw, bh in boxes:
            sanitized = sanitize_box(xc, yc, bw, bh)

            if sanitized is None:
                continue

            xc, yc, bw, bh = sanitized
            f.write(f"{int(cls)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")


def collect_images(img_dir: Path):
    """
    Collects supported image files.
    """
    return sorted([p for p in Path(img_dir).iterdir() if p.suffix.lower() in IMAGE_EXTS])


# ============================================================
# Augmentation policy
# ============================================================

geometric_transforms = [
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.06,
        scale_limit=0.15,
        rotate_limit=12,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.60,
    ),
]

if USE_STRONG_ORIENTATION_AUG:
    geometric_transforms.extend([
        A.VerticalFlip(p=0.30),
        A.RandomRotate90(p=0.30),
    ])

photometric_transforms = [
    A.OneOf(
        [
            A.RandomBrightnessContrast(
                brightness_limit=0.18,
                contrast_limit=0.18,
                p=1.0,
            ),
            A.CLAHE(
                clip_limit=2.0,
                tile_grid_size=(8, 8),
                p=1.0,
            ),
        ],
        p=0.55,
    ),
    A.HueSaturationValue(
        hue_shift_limit=8,
        sat_shift_limit=25,
        val_shift_limit=18,
        p=0.35,
    ),
    A.OneOf(
        [
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
        ],
        p=0.18,
    ),
]

train_transform = A.Compose(
    geometric_transforms + photometric_transforms,
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.25,
        min_area=4,
        check_each_transform=True,
    ),
)

print("=" * 70)
print("Augmentation Policy")
print("=" * 70)
print(train_transform)


# ============================================================
# Build augmented split
# ============================================================

if AUG_SPLIT.exists():
    shutil.rmtree(AUG_SPLIT)

AUG_SPLIT.mkdir(parents=True, exist_ok=True)

audit_rows = []


def copy_split_with_clean_labels(split_name: str):
    """
    Copies original images and sanitized label files into AUG_SPLIT.
    This does not mutate the original SPLIT directory.
    """
    src_img_dir = SPLIT / split_name / "images"
    src_lbl_dir = SPLIT / split_name / "labels"

    dst_img_dir = AUG_SPLIT / split_name / "images"
    dst_lbl_dir = AUG_SPLIT / split_name / "labels"

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    img_files = collect_images(src_img_dir)

    split_stats = Counter()
    split_class_counter = Counter()

    print(f"\n[{split_name}] Copying originals with sanitized labels: {len(img_files)} images")

    for img_path in tqdm(img_files):
        stem = img_path.stem
        lbl_path = src_lbl_dir / f"{stem}.txt"

        dst_img_path = dst_img_dir / img_path.name
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"

        # Copy image
        shutil.copy2(img_path, dst_img_path)
        split_stats["original_images_copied"] += 1

        # Load, clean, and save labels into AUG_SPLIT only
        boxes, label_stats = load_yolo_labels_safe(lbl_path, num_classes)

        for k, v in label_stats.items():
            split_stats[k] += v

        for cls, *_ in boxes:
            split_class_counter[cls] += 1

        save_yolo_labels(dst_lbl_path, boxes)
        split_stats["original_boxes_saved"] += len(boxes)

    return img_files, split_stats, split_class_counter


def augment_training_images(train_img_files):
    """
    Creates augmented copies for training images only.
    The source training images are sampled in shuffled order to avoid
    augmentation bias toward alphabetically early files.
    """
    src_lbl_dir = SPLIT / "train" / "labels"
    dst_img_dir = AUG_SPLIT / "train" / "images"
    dst_lbl_dir = AUG_SPLIT / "train" / "labels"

    shuffled = list(train_img_files)
    random.Random(SEED).shuffle(shuffled)

    aug_stats = Counter()
    aug_class_counter = Counter()

    print(f"\n[train] Generating up to {MAX_AUG_IMAGES} augmented images")

    for img_path in tqdm(shuffled):
        if aug_stats["augmented_images_saved"] >= MAX_AUG_IMAGES:
            break

        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"

        boxes, label_stats = load_yolo_labels_safe(lbl_path, num_classes)

        if len(boxes) == 0:
            aug_stats["skipped_no_valid_boxes"] += 1
            continue

        image_bgr = cv2.imread(str(img_path))

        if image_bgr is None:
            aug_stats["skipped_unreadable_images"] += 1
            continue

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        bboxes = [[b[1], b[2], b[3], b[4]] for b in boxes]
        class_labels = [int(b[0]) for b in boxes]

        for aug_idx in range(N_AUG_PER_IMAGE):
            if aug_stats["augmented_images_saved"] >= MAX_AUG_IMAGES:
                break

            aug_stats["augmentation_attempts"] += 1

            try:
                augmented = train_transform(
                    image=image_rgb,
                    bboxes=bboxes,
                    class_labels=class_labels,
                )
            except Exception:
                aug_stats["augmentation_failures"] += 1
                continue

            aug_img = augmented["image"]
            aug_bboxes = augmented["bboxes"]
            aug_labels = augmented["class_labels"]

            if len(aug_bboxes) == 0:
                aug_stats["dropped_all_boxes_after_aug"] += 1
                continue

            new_boxes = []

            for cls, bbox in zip(aug_labels, aug_bboxes):
                xc, yc, bw, bh = bbox
                sanitized = sanitize_box(xc, yc, bw, bh)

                if sanitized is None:
                    aug_stats["invalid_augmented_boxes"] += 1
                    continue

                xc, yc, bw, bh = sanitized
                cls = int(cls)
                new_boxes.append((cls, xc, yc, bw, bh))
                aug_class_counter[cls] += 1

            if len(new_boxes) == 0:
                aug_stats["dropped_all_boxes_after_sanitize"] += 1
                continue

            new_stem = f"{img_path.stem}_aug{aug_idx + 1}"
            out_img_path = dst_img_dir / f"{new_stem}.jpg"
            out_lbl_path = dst_lbl_dir / f"{new_stem}.txt"

            aug_img_bgr = cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            write_ok = cv2.imwrite(str(out_img_path), aug_img_bgr)

            if not write_ok:
                aug_stats["image_write_failures"] += 1
                continue

            save_yolo_labels(out_lbl_path, new_boxes)

            aug_stats["augmented_images_saved"] += 1
            aug_stats["augmented_boxes_saved"] += len(new_boxes)

    return aug_stats, aug_class_counter


# Copy all splits first; no augmentation is applied to validation/test
split_image_files = {}
split_copy_stats = {}
split_class_counts = {}

for split_name in ["train", "valid", "test"]:
    img_files, stats, cls_counter = copy_split_with_clean_labels(split_name)
    split_image_files[split_name] = img_files
    split_copy_stats[split_name] = stats
    split_class_counts[split_name] = cls_counter

# Training-only augmentation
aug_stats, aug_class_counter = augment_training_images(split_image_files["train"])

# Combine audit rows
for split_name in ["train", "valid", "test"]:
    row = {
        "split": split_name,
        "original_images_copied": split_copy_stats[split_name]["original_images_copied"],
        "original_boxes_saved": split_copy_stats[split_name]["original_boxes_saved"],
        "missing_label_files": split_copy_stats[split_name]["missing_label_file"],
        "invalid_label_lines": split_copy_stats[split_name]["invalid_label_lines"],
        "invalid_class_ids": split_copy_stats[split_name]["invalid_class_ids"],
        "invalid_boxes_removed": split_copy_stats[split_name]["invalid_boxes"],
        "augmented_images_saved": 0,
        "augmented_boxes_saved": 0,
        "augmentation_attempts": 0,
        "dropped_all_boxes_after_aug": 0,
    }

    if split_name == "train":
        row.update({
            "augmented_images_saved": aug_stats["augmented_images_saved"],
            "augmented_boxes_saved": aug_stats["augmented_boxes_saved"],
            "augmentation_attempts": aug_stats["augmentation_attempts"],
            "dropped_all_boxes_after_aug": aug_stats["dropped_all_boxes_after_aug"],
            "augmentation_failures": aug_stats["augmentation_failures"],
            "skipped_no_valid_boxes": aug_stats["skipped_no_valid_boxes"],
            "skipped_unreadable_images": aug_stats["skipped_unreadable_images"],
        })
    else:
        row.update({
            "augmentation_failures": 0,
            "skipped_no_valid_boxes": 0,
            "skipped_unreadable_images": 0,
        })

    audit_rows.append(row)

df_augmentation_audit = pd.DataFrame(audit_rows)

print("=" * 70)
print("Augmentation Audit")
print("=" * 70)
display(df_augmentation_audit)

# -----------------------------
# Write augmented data YAML
# -----------------------------
cfg_aug = cfg.copy()
cfg_aug["path"]  = str(AUG_SPLIT)
cfg_aug["train"] = "train/images"
cfg_aug["val"]   = "valid/images"
cfg_aug["test"]  = "test/images"

with open(DATA_AUG, "w") as f:
    yaml.safe_dump(cfg_aug, f, sort_keys=False)

print("=" * 70)
print("Augmented Dataset YAML")
print("=" * 70)
print("Augmented YAML written to:", DATA_AUG)
print(yaml.safe_dump(cfg_aug, sort_keys=False))

print("=" * 70)
print("Augmentation Completed")
print("=" * 70)
print("Important: augmentation was applied only to the training split.")
print("Validation and test images were copied without augmentation.")

## Cell 6 — Dataset Size and Class Distribution

In [ ]:
# ============================================================
# Cell 6 — Dataset Size and Class Distribution Audit
# ============================================================

from pathlib import Path
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import yaml

BASE  = Path("/kaggle/working")
WORK  = BASE / "palm_byol_ssl"
SPLIT = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

with open(DATA_AUG, "r") as f:
    cfg_aug = yaml.safe_load(f)

class_names = cfg_aug["names"]


def count_images(root_dir: Path):
    counts = {}

    for split in ["train", "valid", "test"]:
        img_dir = root_dir / split / "images"

        if not img_dir.exists():
            counts[split] = 0
        else:
            counts[split] = len(
                [p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
            )

    return counts


def count_labels_and_boxes(root_dir: Path, class_names):
    rows = []

    for split in ["train", "valid", "test"]:
        img_dir = root_dir / split / "images"
        lbl_dir = root_dir / split / "labels"

        img_count = len([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else 0
        label_files = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

        cls_counter = Counter()
        total_boxes = 0
        empty_label_files = 0

        for lbl_path in label_files:
            lines = [ln.strip() for ln in lbl_path.read_text().splitlines() if ln.strip()]

            if len(lines) == 0:
                empty_label_files += 1

            for line in lines:
                parts = line.split()

                if len(parts) != 5:
                    continue

                try:
                    cls_id = int(float(parts[0]))
                except Exception:
                    continue

                cls_counter[cls_id] += 1
                total_boxes += 1

        for cls_id, cls_name in enumerate(class_names):
            rows.append({
                "split": split,
                "class_id": cls_id,
                "class_name": cls_name,
                "images": img_count,
                "label_files": len(label_files),
                "empty_label_files": empty_label_files,
                "instances": cls_counter.get(cls_id, 0),
                "total_boxes": total_boxes,
            })

    return pd.DataFrame(rows)


before = count_images(SPLIT)
after  = count_images(AUG_SPLIT)

df_image_counts = pd.DataFrame([
    {
        "split": split,
        "before_images": before[split],
        "after_images": after[split],
        "increase": after[split] - before[split],
    }
    for split in ["train", "valid", "test"]
])

print("=" * 70)
print("Image Counts Before vs After Augmentation")
print("=" * 70)
display(df_image_counts)

df_aug_distribution = count_labels_and_boxes(AUG_SPLIT, class_names)

print("=" * 70)
print("Class Distribution in Augmented YOLO Dataset")
print("=" * 70)
display(df_aug_distribution)

# -----------------------------
# Publication-ready bar plot
# -----------------------------
plot_df = df_aug_distribution.pivot(
    index="class_name",
    columns="split",
    values="instances"
).fillna(0)

fig, ax = plt.subplots(figsize=(9, 5), dpi=160)

plot_df[["train", "valid", "test"]].plot(kind="bar", ax=ax)

ax.set_title(
    "Class Distribution After Training-Only Augmentation",
    fontsize=14,
    fontweight="bold"
)
ax.set_xlabel("Class")
ax.set_ylabel("Number of Object Instances")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Split")
ax.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()

fig_path = FIG_DIR / "class_distribution_after_augmentation.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Class distribution figure saved to:", fig_path)

## Cell 7 — Visual Sanity Check of Augmented Training Samples

In [ ]:
# ============================================================
# Cell 7 — Visual Sanity Check of Augmented Training Samples
# ============================================================

from pathlib import Path
import random
import math
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

split = "train"

img_dir = AUG_SPLIT / split / "images"
lbl_dir = AUG_SPLIT / split / "labels"

with open(DATA_AUG, "r") as f:
    cfg = yaml.safe_load(f)

class_names = cfg["names"]


def load_yolo_labels_for_vis(lbl_path):
    boxes = []

    if not lbl_path.exists():
        return boxes

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                cls = int(float(parts[0]))
                x_c, y_c, w, h = map(float, parts[1:])
            except Exception:
                continue

            if w <= 0 or h <= 0:
                continue

            boxes.append((cls, x_c, y_c, w, h))

    return boxes


def yolo_box_to_xyxy(xc, yc, bw, bh, img_w, img_h):
    x1 = int(round((xc - bw / 2) * img_w))
    y1 = int(round((yc - bh / 2) * img_h))
    x2 = int(round((xc + bw / 2) * img_w))
    y2 = int(round((yc + bh / 2) * img_h))

    x1 = max(0, min(x1, img_w - 1))
    y1 = max(0, min(y1, img_h - 1))
    x2 = max(0, min(x2, img_w - 1))
    y2 = max(0, min(y2, img_h - 1))

    if x2 <= x1 or y2 <= y1:
        return None

    return x1, y1, x2, y2


def stable_class_colors(num_classes):
    rng = np.random.default_rng(SEED)
    colors = rng.integers(40, 230, size=(num_classes, 3), dtype=np.uint8)
    return [tuple(map(int, c)) for c in colors]


CLASS_COLORS = stable_class_colors(len(class_names))


def draw_boxes(img_path, lbl_path):
    img = cv2.imread(str(img_path))

    if img is None:
        return None, []

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    boxes = load_yolo_labels_for_vis(lbl_path)
    drawn_labels = []

    for cls, xc, yc, bw, bh in boxes:
        xyxy = yolo_box_to_xyxy(xc, yc, bw, bh, img_w, img_h)

        if xyxy is None:
            continue

        x1, y1, x2, y2 = xyxy

        color = CLASS_COLORS[cls] if cls < len(CLASS_COLORS) else (255, 0, 0)
        name = class_names[cls] if cls < len(class_names) else str(cls)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        cv2.putText(
            img,
            name,
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA
        )

        drawn_labels.append(name)

    return img, drawn_labels


all_imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

# Prefer displaying actual augmented files first
aug_imgs = [p for p in all_imgs if "_aug" in p.stem]
candidate_imgs = aug_imgs if len(aug_imgs) > 0 else all_imgs

n_show = min(8, len(candidate_imgs))

if n_show == 0:
    raise RuntimeError(f"No images found for visualization in: {img_dir}")

sample_imgs = random.Random(SEED).sample(candidate_imgs, n_show)

cols = 4
rows = math.ceil(n_show / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 4.8 * rows), dpi=160)
axes = np.array(axes).reshape(-1) if rows * cols > 1 else np.array([axes])

summary_rows = []

for i, ax in enumerate(axes):
    ax.axis("off")

    if i >= n_show:
        continue

    img_path = sample_imgs[i]
    lbl_path = lbl_dir / f"{img_path.stem}.txt"

    vis, drawn_labels = draw_boxes(img_path, lbl_path)

    if vis is None:
        ax.set_title(f"Unreadable\n{img_path.name}", fontsize=9)
        continue

    ax.imshow(vis)
    ax.set_title(
        f"{img_path.name[:32]}\nBoxes: {len(drawn_labels)}",
        fontsize=8
    )

    summary_rows.append({
        "image": img_path.name,
        "num_boxes": len(drawn_labels),
        "classes": ", ".join(sorted(set(drawn_labels))) if drawn_labels else "None",
        "is_augmented": "_aug" in img_path.stem,
    })

fig.suptitle(
    "Visual Sanity Check of Augmented Training Samples",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.94])

fig_path = FIG_DIR / "augmented_training_samples_sanity_check.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("=" * 70)
print("Augmented Sample Visualization Summary")
print("=" * 70)
display(pd.DataFrame(summary_rows))
print("Figure saved to:", fig_path)

## Cell 8 — BYOL + YOLOv12s Experiment Configuration

In [ ]:
# ============================================================
# Cell 8 — BYOL + YOLOv12s Experiment Configuration
# ============================================================

import os
import gc
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

from ultralytics import YOLO
from IPython.display import Image as IPyImage, display

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

# Original split + YAML
SPLIT = WORK / "0_yolo_split"
DATA  = WORK / "data_byol.yaml"

# Augmented split + YAML
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG  = WORK / "data_byol_augmented.yaml"

# BYOL SSL weights for YOLOv12s backbone/neck
SSL_W = WORK / "byol_backbone_with_hooks_yolov12s.pt"

# YOLOv12s model config
MODEL_CFG = "yolov12s.yaml"

# Output folders
FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

print("=" * 70)
print("Experiment Configuration")
print("=" * 70)
print("Device              :", device)
if device == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
print("Original split       :", SPLIT)
print("Original data YAML   :", DATA)
print("Augmented split      :", AUG_SPLIT)
print("Augmented data YAML  :", DATA_AUG)
print("BYOL SSL weights     :", SSL_W)
print("YOLOv12s config      :", MODEL_CFG)
print("Figure directory     :", FIG_DIR)
print("Results directory    :", RESULTS_DIR)

# -----------------------------
# Safety checks
# -----------------------------
required_paths = {
    "SPLIT": SPLIT,
    "DATA": DATA,
    "AUG_SPLIT": AUG_SPLIT,
    "DATA_AUG": DATA_AUG,
}

for name, path in required_paths.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

print("\nAll required dataset paths verified.")

## Cell 9 — BYOL SSL Dataset and Training-Only Augmentations

In [ ]:
# ============================================================
# Cell 9 — BYOL SSL Dataset and Training-Only Augmentations
# ============================================================

from pathlib import Path
import os
import random
import warnings
from PIL import Image, ImageFile

import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

# -----------------------------
# Paths and reproducibility
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# -----------------------------
# BYOL image size
# -----------------------------
BYOL_IMG_SIZE = 160

# ------------------------------------------------------------
# BYOL-style augmentations
# ------------------------------------------------------------
# Original scale=(0.2, 1.0) is common in natural-image SSL,
# but may crop out small object evidence in object-detection data.
# This version is still strong, but less destructive for palm detection.
byol_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        BYOL_IMG_SIZE,
        scale=(0.45, 1.0),
        ratio=(0.75, 1.333),
        antialias=True,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply(
        [transforms.ColorJitter(
            brightness=0.35,
            contrast=0.35,
            saturation=0.30,
            hue=0.08,
        )],
        p=0.75,
    ),
    transforms.RandomGrayscale(p=0.15),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))],
        p=0.20,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class TwoView(Dataset):
    """
    Returns two independently augmented views of the same image for BYOL.

    Methodological safeguard:
    image_dirs must contain training images only.
    Validation and test images must not be used here.
    """

    def __init__(self, image_dirs, transform, verify_images=True):
        self.image_paths = []

        for d in image_dirs:
            d = Path(d)

            if not d.exists():
                raise FileNotFoundError(f"SSL image directory not found: {d}")

            self.image_paths.extend(
                sorted([p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTS])
            )

        self.image_paths = sorted(list(dict.fromkeys(self.image_paths)))
        self.transform = transform

        if len(self.image_paths) == 0:
            raise RuntimeError("No SSL images found. Check ssl_image_dirs.")

        if verify_images:
            valid_paths = []
            corrupt_paths = []

            for p in tqdm(self.image_paths, desc="Verifying SSL images", leave=False):
                try:
                    with Image.open(p) as img:
                        img.verify()
                    valid_paths.append(p)
                except Exception:
                    corrupt_paths.append(p)

            self.image_paths = valid_paths

            if corrupt_paths:
                print(f"WARNING: {len(corrupt_paths)} corrupt SSL images were excluded.")

        if len(self.image_paths) < 2:
            raise RuntimeError("BYOL requires at least 2 valid training images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to read SSL image: {img_path}") from e

        v1 = self.transform(img)
        v2 = self.transform(img)

        return v1, v2


# ------------------------------------------------------------
# Important leakage fix:
# Use training images only for SSL pretraining.
# Do NOT include validation or test images.
# ------------------------------------------------------------
ssl_image_dirs = [
    AUG_SPLIT / "train" / "images"
]

print("=" * 70)
print("BYOL SSL Dataset Configuration")
print("=" * 70)
print("BYOL image size:", BYOL_IMG_SIZE)
print("SSL image dirs:")
for d in ssl_image_dirs:
    print(" -", d)

print("\nLeakage safeguard: validation/test images are NOT used for SSL pretraining.")

## Cell 10 — BYOL Loss and YOLO Feature-Hook Utilities

In [ ]:
# ============================================================
# Cell 10 — BYOL Loss and YOLO Feature-Hook Utilities
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


def byol_loss(p, z):
    """
    BYOL negative cosine similarity loss.

    Args:
        p: online prediction, shape [B, D]
        z: target projection, shape [B, D]

    Returns:
        scalar loss
    """
    p = F.normalize(p, dim=1)
    z = F.normalize(z.detach(), dim=1)
    return 2.0 - 2.0 * (p * z).sum(dim=1).mean()


class DetectInputHook:
    """
    Forward pre-hook used to capture the multi-scale feature maps
    entering the YOLO Detect head.

    For YOLO detectors, this usually captures P3/P4/P5 features.
    """

    def __init__(self, detect_module):
        if detect_module is None:
            raise ValueError("Detect module is None. Could not register feature hook.")

        self.feats = None
        self.handle = detect_module.register_forward_pre_hook(self.hook)

    def hook(self, module, inputs):
        if len(inputs) == 0:
            self.feats = None
            return None

        x = inputs[0]

        if isinstance(x, (list, tuple)):
            self.feats = list(x)
        else:
            self.feats = [x]

        return None

    def close(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


def global_pool_concat(feats):
    """
    Global-average pools and concatenates multi-scale feature maps.

    Args:
        feats: list of tensors, each [B, C_i, H_i, W_i]

    Returns:
        Tensor [B, sum(C_i)]
    """
    if feats is None or len(feats) == 0:
        raise RuntimeError("No feature maps were captured from the YOLO Detect hook.")

    pooled = []

    for f in feats:
        if not torch.is_tensor(f):
            raise TypeError("Captured feature is not a tensor.")

        if f.ndim != 4:
            raise ValueError(f"Expected feature map shape [B,C,H,W], got {tuple(f.shape)}")

        pooled.append(F.adaptive_avg_pool2d(f, 1).flatten(1))

    return torch.cat(pooled, dim=1)


def get_detect_module(model_module):
    """
    Robustly returns the YOLO Detect head from an Ultralytics DetectionModel.
    """
    try:
        last = model_module.model[-1]
        if "Detect" in last.__class__.__name__:
            return last
    except Exception:
        pass

    candidates = []

    for m in model_module.modules():
        if "Detect" in m.__class__.__name__:
            candidates.append(m)

    if len(candidates) == 0:
        return None

    return candidates[-1]


def get_detect_layer_index(model_module):
    """
    Returns the index of the Detect layer if it is the final model layer.
    Used to exclude detection-head weights when saving SSL backbone weights.
    """
    try:
        for idx, layer in enumerate(model_module.model):
            if "Detect" in layer.__class__.__name__:
                return idx
    except Exception:
        pass

    return None


@torch.no_grad()
def ema_update(src_module, dst_module, momentum):
    """
    Momentum update:
        target = m * target + (1 - m) * online
    """
    for src_p, dst_p in zip(src_module.parameters(), dst_module.parameters()):
        dst_p.data.mul_(momentum).add_(src_p.data, alpha=1.0 - momentum)


def momentum_scheduled(step, total_steps, base_m=0.996):
    """
    Cosine schedule for BYOL EMA momentum.
    Increases m from base_m toward 1.0.
    """
    if total_steps <= 1:
        return 1.0

    tau = step / float(total_steps - 1)
    return 1.0 - (1.0 - base_m) * (0.5 * (1.0 + math.cos(math.pi * tau)))


def build_mlp_projector(feat_dim, hidden_dim=512, out_dim=256):
    """
    BYOL projector MLP.
    """
    return nn.Sequential(
        nn.Linear(feat_dim, hidden_dim, bias=False),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, out_dim, bias=False),
    )


def build_mlp_predictor(in_dim=256, hidden_dim=512, out_dim=256):
    """
    BYOL predictor MLP.
    """
    return nn.Sequential(
        nn.Linear(in_dim, hidden_dim, bias=False),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, out_dim, bias=False),
    )

if SSL_W.exists():
    SSL_W.unlink()
    print("Deleted old SSL checkpoint:", SSL_W)

## Cell 11 — BYOL Pretraining for YOLOv12s Backbone/Neck

In [ ]:
# ============================================================
# Cell 11 — BYOL Pretraining for YOLOv12s Backbone/Neck
# ============================================================

import os
import gc
import json
import math
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from ultralytics import YOLO
from tqdm.auto import tqdm

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# BYOL SSL weights for YOLOv12s backbone/neck
SSL_W = WORK / "byol_backbone_with_hooks_yolov12s.pt"

# Model source.
# Ultralytics YOLOv12 commonly uses yolo12s.pt naming.
# The fallback list keeps this robust across notebook versions.
MODEL_CFG = globals().get("MODEL_CFG", "yolo12s.pt")

MODEL_CANDIDATES = [
    MODEL_CFG,
    "yolo12s.pt",
    "yolov12s.pt",
    "yolo12s.yaml",
    "yolov12s.yaml",
]

# BYOL pretraining hyperparameters
SSL_EPOCHS = 70
SSL_BATCH = 8
SSL_LR = 1e-3
SSL_WEIGHT_DECAY = 1e-4
SSL_BASE_MOMENTUM = 0.996

# Safer for Kaggle/Jupyter.
# This avoids repeated multiprocessing cleanup warnings.
SSL_NUM_WORKERS = 0

# Early stopping for SSL pretraining
SSL_EARLY_STOP_PATIENCE = 10
SSL_MIN_DELTA = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
is_cuda = device == "cuda"

if is_cuda:
    torch.cuda.empty_cache()

print("=" * 70)
print("BYOL Pretraining Configuration")
print("=" * 70)
print("Device             :", device)
if is_cuda:
    print("GPU                :", torch.cuda.get_device_name(0))
print("SSL weights output :", SSL_W)
print("Requested model    :", MODEL_CFG)
print("SSL epochs         :", SSL_EPOCHS)
print("SSL batch size     :", SSL_BATCH)
print("SSL learning rate  :", SSL_LR)
print("SSL weight decay   :", SSL_WEIGHT_DECAY)
print("Base EMA momentum  :", SSL_BASE_MOMENTUM)
print("SSL num workers    :", SSL_NUM_WORKERS)
print("Early stopping     : Enabled")
print("Early stop patience:", SSL_EARLY_STOP_PATIENCE)
print("Early stop min delta:", SSL_MIN_DELTA)


def build_yolo_detection_model(model_candidates, device):
    """
    Builds an Ultralytics YOLO model from the first working candidate.
    """
    tried = []
    seen = set()

    for candidate in model_candidates:
        if candidate is None:
            continue

        candidate = str(candidate)

        if candidate in seen:
            continue

        seen.add(candidate)
        tried.append(candidate)

        try:
            yolo = YOLO(candidate)
            model = yolo.model.to(device)
            return model, candidate

        except Exception as e:
            print(f"Could not initialize YOLO model from '{candidate}': {type(e).__name__}: {e}")

    raise RuntimeError(f"Could not initialize YOLO model. Tried: {tried}")


def save_backbone_without_detect(model_module, save_path):
    """
    Saves YOLO backbone/neck weights while excluding the Detect head.
    """
    detect_idx = get_detect_layer_index(model_module)

    if detect_idx is None:
        print("WARNING: Detect head index could not be identified. Saving full state_dict.")
        state = {k: v.detach().cpu() for k, v in model_module.state_dict().items()}

    else:
        state = {
            k: v.detach().cpu()
            for k, v in model_module.state_dict().items()
            if not k.startswith(f"model.{detect_idx}.")
        }

    torch.save(state, save_path)
    return len(state)


def set_detect_head_train_mode(detect_module):
    """
    Important for BYOL feature extraction.

    In eval mode, YOLO Detect head performs full inference decoding,
    including anchor generation. For BYOL, we only need the feature maps
    entering Detect. So the Detect head should remain in training mode
    to avoid the inference/decode path.

    This does not change path names or variable names.
    """
    if detect_module is not None:
        detect_module.train()


def forward_for_hook(model_module, detect_module, x):
    """
    Forward pass only to trigger DetectInputHook.

    The Detect head is forced to training mode to avoid YOLO inference decoding.
    """
    set_detect_head_train_mode(detect_module)
    return model_module(x)


# ------------------------------------------------------------
# Skip logic
# ------------------------------------------------------------
if SSL_W.exists():
    print("=" * 70)
    print("Cached BYOL Backbone Found")
    print("=" * 70)
    print("Skipping BYOL pretraining:", SSL_W)

else:
    print("=" * 70)
    print("Starting BYOL Pretraining")
    print("=" * 70)
    print("SSL data source: TRAINING IMAGES ONLY")
    for d in ssl_image_dirs:
        print(" -", d)

    # -----------------------------
    # Dataset and DataLoader
    # -----------------------------
    ds = TwoView(
        image_dirs=ssl_image_dirs,
        transform=byol_transform,
        verify_images=True,
    )

    if len(ds) <= SSL_BATCH:
        raise RuntimeError(
            f"SSL dataset has {len(ds)} images but SSL_BATCH={SSL_BATCH}. "
            "Increase data or reduce SSL_BATCH while keeping batch size > 1."
        )

    def seed_worker(worker_id):
        worker_seed = SEED + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    generator = torch.Generator()
    generator.manual_seed(SEED)

    dl = DataLoader(
        ds,
        batch_size=SSL_BATCH,
        shuffle=True,
        num_workers=SSL_NUM_WORKERS,
        pin_memory=is_cuda,
        drop_last=True,
        worker_init_fn=seed_worker,
        generator=generator,
    )

    if len(dl) == 0:
        raise RuntimeError("BYOL DataLoader has zero batches. Check SSL_BATCH and dataset size.")

    print(f"SSL images used: {len(ds)}")
    print(f"SSL batches/epoch: {len(dl)}")

    # -----------------------------
    # Build online and target models
    # -----------------------------
    online_full, resolved_model_source = build_yolo_detection_model(MODEL_CANDIDATES, device)
    target_full, _ = build_yolo_detection_model([resolved_model_source], device)

    target_full.load_state_dict(online_full.state_dict(), strict=True)

    for p in target_full.parameters():
        p.requires_grad = False

    print("Resolved YOLO model source:", resolved_model_source)

    # -----------------------------
    # Register hooks
    # -----------------------------
    online_detect = get_detect_module(online_full)
    target_detect = get_detect_module(target_full)

    if online_detect is None or target_detect is None:
        raise RuntimeError("Could not locate YOLO Detect module for feature extraction.")

    online_hook = DetectInputHook(online_detect)
    target_hook = DetectInputHook(target_detect)

    # -----------------------------
    # Infer feature dimension
    # -----------------------------
    online_full.eval()
    set_detect_head_train_mode(online_detect)

    with torch.no_grad():
        dummy = torch.zeros(1, 3, BYOL_IMG_SIZE, BYOL_IMG_SIZE, device=device)
        _ = forward_for_hook(online_full, online_detect, dummy)
        feat_dim = global_pool_concat(online_hook.feats).shape[1]

    print("Backbone/neck pooled feature dimension:", feat_dim)

    # -----------------------------
    # Projector and predictor
    # -----------------------------
    proj_o = build_mlp_projector(feat_dim).to(device)
    proj_t = build_mlp_projector(feat_dim).to(device)
    pred_o = build_mlp_predictor().to(device)

    proj_t.load_state_dict(proj_o.state_dict(), strict=True)

    for p in proj_t.parameters():
        p.requires_grad = False

    # -----------------------------
    # Optimizer and AMP
    # -----------------------------
    opt = torch.optim.AdamW(
        list(online_full.parameters()) +
        list(proj_o.parameters()) +
        list(pred_o.parameters()),
        lr=SSL_LR,
        weight_decay=SSL_WEIGHT_DECAY,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=is_cuda)

    # -----------------------------
    # Logging containers
    # -----------------------------
    ssl_step_losses = []
    ssl_epoch_losses = []
    ssl_momentum = []
    ssl_lr_values = []

    total_steps = SSL_EPOCHS * len(dl)
    global_step = 0

    best_epoch_loss = float("inf")
    best_epoch = -1

    epochs_without_improvement = 0
    early_stopped = False
    early_stop_epoch = None

    ssl_log_rows = []

    start_time = time.time()

    # -----------------------------
    # Training loop
    # -----------------------------
    for ep in range(SSL_EPOCHS):
        online_full.train()
        proj_o.train()
        pred_o.train()

        # Target network stays eval, but Detect head is forced to train mode
        # to avoid YOLO inference decoding during BYOL feature extraction.
        target_full.eval()
        proj_t.eval()
        set_detect_head_train_mode(target_detect)

        running = 0.0
        n_batches = 0

        pbar = tqdm(dl, desc=f"BYOL {ep + 1}/{SSL_EPOCHS}", leave=False)

        for v1, v2 in pbar:
            v1 = v1.to(device, non_blocking=is_cuda)
            v2 = v2.to(device, non_blocking=is_cuda)

            m_cur = momentum_scheduled(
                global_step,
                total_steps,
                base_m=SSL_BASE_MOMENTUM,
            )

            opt.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", enabled=is_cuda):
                # Online branch
                _ = forward_for_hook(online_full, online_detect, v1)
                h1 = global_pool_concat(online_hook.feats)

                _ = forward_for_hook(online_full, online_detect, v2)
                h2 = global_pool_concat(online_hook.feats)

                z1_o = proj_o(h1)
                z2_o = proj_o(h2)

                p1 = pred_o(z1_o)
                p2 = pred_o(z2_o)

                # Target branch: no gradients
                with torch.no_grad():
                    target_full.eval()
                    set_detect_head_train_mode(target_detect)

                    _ = forward_for_hook(target_full, target_detect, v1)
                    h1_t = global_pool_concat(target_hook.feats)

                    _ = forward_for_hook(target_full, target_detect, v2)
                    h2_t = global_pool_concat(target_hook.feats)

                    z1_t = proj_t(h1_t)
                    z2_t = proj_t(h2_t)

                loss = byol_loss(p1, z2_t) + byol_loss(p2, z1_t)

            if not torch.isfinite(loss):
                print(f"WARNING: Non-finite BYOL loss at step {global_step}. Skipping batch.")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)

            torch.nn.utils.clip_grad_norm_(
                list(online_full.parameters()) +
                list(proj_o.parameters()) +
                list(pred_o.parameters()),
                max_norm=5.0,
            )

            scaler.step(opt)
            scaler.update()

            # Correct BYOL order:
            # update target networks AFTER online optimizer step.
            with torch.no_grad():
                ema_update(online_full, target_full, m_cur)
                ema_update(proj_o, proj_t, m_cur)

            loss_val = float(loss.item())
            running += loss_val
            n_batches += 1

            current_lr = opt.param_groups[0]["lr"]

            ssl_step_losses.append(loss_val)
            ssl_momentum.append(float(m_cur))
            ssl_lr_values.append(float(current_lr))

            ssl_log_rows.append({
                "epoch": ep + 1,
                "step": global_step + 1,
                "loss": loss_val,
                "ema_momentum": float(m_cur),
                "lr": float(current_lr),
            })

            global_step += 1

            pbar.set_postfix(
                loss=f"{loss_val:.4f}",
                m=f"{m_cur:.5f}",
                lr=f"{current_lr:.2e}",
            )

        epoch_loss = running / max(1, n_batches)
        ssl_epoch_losses.append(epoch_loss)

        print(f"Epoch {ep + 1:03d}/{SSL_EPOCHS} | BYOL loss: {epoch_loss:.6f}")

        # ----------------------------------------------------
        # Save best SSL checkpoint + early stopping
        # ----------------------------------------------------
        if epoch_loss < (best_epoch_loss - SSL_MIN_DELTA):
            best_epoch_loss = epoch_loss
            best_epoch = ep + 1
            epochs_without_improvement = 0

            n_saved = save_backbone_without_detect(online_full, SSL_W)

            print(
                f"  ✓ Best SSL checkpoint updated at epoch {best_epoch} "
                f"| loss={best_epoch_loss:.6f} | tensors saved={n_saved}"
            )

        else:
            epochs_without_improvement += 1

            print(
                f"  No meaningful SSL improvement for "
                f"{epochs_without_improvement}/{SSL_EARLY_STOP_PATIENCE} epochs "
                f"| best_epoch={best_epoch} | best_loss={best_epoch_loss:.6f}"
            )

            if epochs_without_improvement >= SSL_EARLY_STOP_PATIENCE:
                early_stopped = True
                early_stop_epoch = ep + 1

                print("=" * 70)
                print("Early stopping triggered for BYOL SSL pretraining")
                print("=" * 70)
                print(f"Stopped at epoch : {early_stop_epoch}")
                print(f"Best epoch       : {best_epoch}")
                print(f"Best BYOL loss   : {best_epoch_loss:.6f}")
                break

    elapsed_min = (time.time() - start_time) / 60.0

    # -----------------------------
    # Save SSL logs
    # -----------------------------
    df_ssl_log = pd.DataFrame(ssl_log_rows)
    ssl_log_path = RESULTS_DIR / "byol_ssl_training_log.csv"
    df_ssl_log.to_csv(ssl_log_path, index=False)

    ssl_summary = {
        "resolved_model_source": resolved_model_source,
        "ssl_images_used": len(ds),
        "ssl_epochs": SSL_EPOCHS,
        "ssl_batch": SSL_BATCH,
        "ssl_learning_rate": SSL_LR,
        "ssl_weight_decay": SSL_WEIGHT_DECAY,
        "ssl_base_momentum": SSL_BASE_MOMENTUM,

        "early_stopping_used": True,
        "early_stop_patience": SSL_EARLY_STOP_PATIENCE,
        "early_stop_min_delta": SSL_MIN_DELTA,
        "early_stopped": early_stopped,
        "early_stop_epoch": early_stop_epoch,

        "best_epoch": best_epoch,
        "best_epoch_loss": float(best_epoch_loss),
        "elapsed_minutes": float(elapsed_min),
        "ssl_weight_path": str(SSL_W),
        "leakage_control": "Only augmented training split was used for BYOL SSL pretraining. Validation/test images were excluded.",
    }

    ssl_summary_path = RESULTS_DIR / "byol_ssl_summary.json"

    with open(ssl_summary_path, "w") as f:
        json.dump(ssl_summary, f, indent=4)

    print("=" * 70)
    print("BYOL Pretraining Summary")
    print("=" * 70)
    print(json.dumps(ssl_summary, indent=4))
    print("Training log saved to:", ssl_log_path)
    print("Summary JSON saved to:", ssl_summary_path)
    print("Best SSL backbone/neck weights saved to:", SSL_W)

    # -----------------------------
    # Publication-ready plots
    # -----------------------------
    if len(ssl_step_losses) > 0:
        steps = np.arange(1, len(ssl_step_losses) + 1)

        # Step loss
        fig, ax = plt.subplots(figsize=(9, 5), dpi=160)
        ax.plot(steps, ssl_step_losses, linewidth=1.2)
        ax.set_title("BYOL Self-Supervised Pretraining Loss per Step", fontsize=14, fontweight="bold")
        ax.set_xlabel("Training Step")
        ax.set_ylabel("BYOL Loss")
        ax.grid(True, linestyle="--", alpha=0.35)
        plt.tight_layout()

        fig_path = FIG_DIR / "byol_ssl_loss_per_step.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()
        print("Saved:", fig_path)

        # Epoch loss
        epochs = np.arange(1, len(ssl_epoch_losses) + 1)

        fig, ax = plt.subplots(figsize=(8, 5), dpi=160)
        ax.plot(epochs, ssl_epoch_losses, marker="o", linewidth=1.8)

        if best_epoch > 0:
            ax.axvline(
                best_epoch,
                linestyle="--",
                linewidth=1.3,
                label=f"Best epoch: {best_epoch}"
            )

        if early_stopped and early_stop_epoch is not None:
            ax.axvline(
                early_stop_epoch,
                linestyle=":",
                linewidth=1.5,
                label=f"Early stop: {early_stop_epoch}"
            )

        ax.set_title("BYOL Self-Supervised Pretraining Loss per Epoch", fontsize=14, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Mean BYOL Loss")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.35)
        plt.tight_layout()

        fig_path = FIG_DIR / "byol_ssl_loss_per_epoch.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()
        print("Saved:", fig_path)

        # EMA momentum schedule
        if len(ssl_momentum) == len(ssl_step_losses):
            fig, ax = plt.subplots(figsize=(9, 5), dpi=160)
            ax.plot(steps, ssl_momentum, linewidth=1.5)
            ax.set_title("BYOL EMA Momentum Schedule", fontsize=14, fontweight="bold")
            ax.set_xlabel("Training Step")
            ax.set_ylabel("EMA Momentum")
            ax.grid(True, linestyle="--", alpha=0.35)
            plt.tight_layout()

            fig_path = FIG_DIR / "byol_ssl_momentum_schedule.png"
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            plt.show()
            print("Saved:", fig_path)

    # -----------------------------
    # Cleanup
    # -----------------------------
    try:
        online_hook.close()
        target_hook.close()
    except Exception:
        pass

    del online_full, target_full, proj_o, proj_t, pred_o, dl, ds
    gc.collect()

    if is_cuda:
        torch.cuda.empty_cache()

## Cell 12 — Fine-Tuning YOLOv12s with BYOL Backbone/Neck Weights

In [ ]:
# ===============================================================
# Cell 12 — Fine-Tuning YOLOv12s with BYOL Backbone/Neck Weights
# ===============================================================

import os
import gc
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

DATA = WORK / "data_byol.yaml"
AUG_SPLIT = WORK / "0_yolo_split_aug"
DATA_AUG = WORK / "data_byol_augmented.yaml"

SSL_W = WORK / "byol_backbone_with_hooks_yolov12s.pt"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CFG = globals().get("MODEL_CFG", "yolo12s.pt")

# Important:
# yolo12s.pt is placed first because Cell 11 resolved to yolo12s.pt.
# This avoids the 'yolov12s.yaml does not exist' error.
MODEL_CANDIDATES = [
    "yolo12s.pt",
    MODEL_CFG,
    "yolov12s.pt",
    "yolo12s.yaml",
    "yolov12s.yaml",
]

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()

# -----------------------------
# Safety checks
# -----------------------------
for name, path in {
    "DATA": DATA,
    "DATA_AUG": DATA_AUG,
    "SSL_W": SSL_W,
}.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{name} not found: {path}")

print("=" * 70)
print("Fine-Tuning Configuration")
print("=" * 70)
print("Requested MODEL_CFG :", MODEL_CFG)
print("Device              :", device)
if device == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
print("Training YAML       :", DATA_AUG)
print("Final eval YAML     :", DATA)
print("BYOL weights        :", SSL_W)


# ============================================================
# Helper: robust YOLO initialization
# ============================================================

def build_yolo_detector(model_candidates):
    """
    Builds YOLO detector from the first working model source.
    This prevents crash if MODEL_CFG points to a missing yaml file.
    """
    tried = []
    seen = set()

    for candidate in model_candidates:
        if candidate is None:
            continue

        candidate = str(candidate)

        if candidate in seen:
            continue

        seen.add(candidate)
        tried.append(candidate)

        try:
            model = YOLO(candidate)
            return model, candidate

        except Exception as e:
            print(f"Could not initialize YOLO from '{candidate}': {type(e).__name__}: {e}")

    raise RuntimeError(f"Could not initialize YOLO detector. Tried: {tried}")


def load_byol_weights_safely(det, ssl_weight_path):
    """
    Loads BYOL backbone/neck weights safely.

    strict=False alone can still fail if a key exists but has a different shape.
    This function loads only keys that:
    1. exist in the detector
    2. have matching tensor shapes
    """
    ssl_state = torch.load(ssl_weight_path, map_location="cpu")
    det_state = det.model.state_dict()

    compatible_state = {}
    skipped_shape_mismatch = []
    skipped_missing_in_model = []

    for k, v in ssl_state.items():
        if k not in det_state:
            skipped_missing_in_model.append(k)
            continue

        if det_state[k].shape != v.shape:
            skipped_shape_mismatch.append((k, tuple(v.shape), tuple(det_state[k].shape)))
            continue

        compatible_state[k] = v

    load_info = det.model.load_state_dict(compatible_state, strict=False)

    missing_keys = list(getattr(load_info, "missing_keys", []))
    unexpected_keys = list(getattr(load_info, "unexpected_keys", []))

    return {
        "ssl_state": ssl_state,
        "compatible_state": compatible_state,
        "missing_keys": missing_keys,
        "unexpected_keys": unexpected_keys,
        "skipped_shape_mismatch": skipped_shape_mismatch,
        "skipped_missing_in_model": skipped_missing_in_model,
    }


def save_json_safe(obj, path):
    """
    Saves JSON safely by converting non-serializable objects to strings.
    """
    def convert(o):
        try:
            json.dumps(o)
            return o
        except Exception:
            return str(o)

    clean_obj = {k: convert(v) for k, v in obj.items()}

    with open(path, "w") as f:
        json.dump(clean_obj, f, indent=4)


def train_with_safe_retry(det, train_config):
    """
    Runs YOLO training.

    If some optional augmentation arguments are unsupported by the installed
    Ultralytics version, it retries with a safer reduced config.
    """
    try:
        return det.train(**train_config)

    except Exception as e:
        print("=" * 70)
        print("First training attempt failed.")
        print("=" * 70)
        print("Error type:", type(e).__name__)
        print("Error message:", str(e))
        print("\nRetrying with safer minimal YOLO training configuration...")

        safe_train_config = dict(train_config)

        optional_keys_to_remove = [
            "mosaic",
            "close_mosaic",
            "mixup",
            "copy_paste",
            "erasing",
            "hsv_h",
            "hsv_s",
            "hsv_v",
            "degrees",
            "translate",
            "scale",
            "fliplr",
            "flipud",
        ]

        for k in optional_keys_to_remove:
            safe_train_config.pop(k, None)

        safe_train_config["name"] = str(train_config["name"]) + "_safe_retry"

        retry_config_path = RESULTS_DIR / "yolov12s_byol_finetune_config_safe_retry.json"
        save_json_safe(safe_train_config, retry_config_path)

        print("Safe retry config saved to:", retry_config_path)

        return det.train(**safe_train_config)


# -----------------------------
# Initialize detector and load BYOL weights
# -----------------------------
print("\nInitializing YOLO model...")

det, resolved_model_source = build_yolo_detector(MODEL_CANDIDATES)

print("=" * 70)
print("Resolved YOLO Detector Source")
print("=" * 70)
print("Requested MODEL_CFG :", MODEL_CFG)
print("Using model source  :", resolved_model_source)

print("\nLoading BYOL backbone/neck weights...")

load_report = load_byol_weights_safely(det, SSL_W)

ssl_state = load_report["ssl_state"]
compatible_state = load_report["compatible_state"]
missing_keys = load_report["missing_keys"]
unexpected_keys = load_report["unexpected_keys"]
skipped_shape_mismatch = load_report["skipped_shape_mismatch"]
skipped_missing_in_model = load_report["skipped_missing_in_model"]

print("=" * 70)
print("BYOL Weight Loading Report")
print("=" * 70)
print(f"Total SSL tensors              : {len(ssl_state)}")
print(f"Compatible tensors loaded      : {len(compatible_state)}")
print(f"Missing keys after load        : {len(missing_keys)}")
print(f"Unexpected keys after load     : {len(unexpected_keys)}")
print(f"Skipped missing-in-model keys  : {len(skipped_missing_in_model)}")
print(f"Skipped shape-mismatch keys    : {len(skipped_shape_mismatch)}")

if len(missing_keys) > 0:
    print("\nFirst missing keys:")
    for k in missing_keys[:10]:
        print(" -", k)

if len(unexpected_keys) > 0:
    print("\nFirst unexpected keys:")
    for k in unexpected_keys[:10]:
        print(" -", k)

if len(skipped_shape_mismatch) > 0:
    print("\nFirst shape-mismatch keys:")
    for k, ssl_shape, model_shape in skipped_shape_mismatch[:10]:
        print(f" - {k}: SSL {ssl_shape} vs model {model_shape}")

if len(compatible_state) == 0:
    raise RuntimeError(
        "No BYOL tensors were compatible with the detector. "
        "Check whether Cell 11 and Cell 12 are using the same YOLO model family."
    )


# -----------------------------
# Detection fine-tuning
# -----------------------------
det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

train_config = {
    "data": str(DATA_AUG),
    "epochs": 70,
    "imgsz": 640,
    "batch": 8,
    "project": str(WORK),
    "name": det_run_name,
    "exist_ok": True,
    "device": device_arg,
    "seed": SEED,
    "deterministic": True,

    # Optimizer and learning rate
    "optimizer": "AdamW",
    "lr0": 5e-4,
    "lrf": 0.01,
    "weight_decay": 5e-4,
    "warmup_epochs": 3.0,
    "cos_lr": True,

    # Early stopping
    "patience": 10,

    "verbose": True,
    "plots": True,

    # Controlled online augmentation.
    # Offline augmentation already exists, so YOLO augmentation is reduced.
    "mosaic": 0.40,
    "close_mosaic": 10,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "erasing": 0.0,
    "hsv_h": 0.010,
    "hsv_s": 0.400,
    "hsv_v": 0.250,
    "degrees": 5.0,
    "translate": 0.05,
    "scale": 0.25,
    "fliplr": 0.50,
    "flipud": 0.0,
}

config_path = RESULTS_DIR / "yolov12s_byol_finetune_config.json"

train_config_to_save = dict(train_config)
train_config_to_save["requested_MODEL_CFG"] = str(MODEL_CFG)
train_config_to_save["resolved_model_source"] = str(resolved_model_source)
train_config_to_save["byol_weight_path"] = str(SSL_W)
train_config_to_save["compatible_byol_tensors_loaded"] = len(compatible_state)
train_config_to_save["total_ssl_tensors"] = len(ssl_state)
train_config_to_save["early_stopping_used"] = True
train_config_to_save["early_stopping_patience"] = 10

save_json_safe(train_config_to_save, config_path)

print("=" * 70)
print("Starting BYOL-initialized YOLOv12s fine-tuning")
print("=" * 70)
print("Training config saved to:", config_path)
print("Run directory:", run_dir)
print("Early stopping patience:", train_config["patience"])

train_start = time.time()

train_results = train_with_safe_retry(det, train_config)

train_elapsed_min = (time.time() - train_start) / 60.0

print("=" * 70)
print("Fine-Tuning Completed")
print("=" * 70)
print(f"Elapsed time: {train_elapsed_min:.2f} minutes")
print("Run directory:", run_dir)

gc.collect()

if device == "cuda":
    torch.cuda.empty_cache()

## Cell 13 — Official Validation/Test Evaluation on Original Split

In [ ]:
# ================================================================
# Cell 13 — Official Validation/Test Evaluation on Original Split
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from IPython.display import Image as IPyImage, display

BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

DATA = WORK / "data_byol.yaml"
DATA_AUG = WORK / "data_byol_augmented.yaml"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

if best_pt.exists():
    model_pt = best_pt
elif last_pt.exists():
    model_pt = last_pt
else:
    raise FileNotFoundError(f"No trained YOLO weights found in: {run_dir / 'weights'}")

best_model = YOLO(str(model_pt))

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

print("=" * 70)
print("Official Evaluation Configuration")
print("=" * 70)
print("Model weights        :", model_pt)
print("Evaluation data YAML :", DATA)
print("Device               :", device)
print("Important            : Final validation/test use the original non-augmented split.")


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def get_attr_safe(obj, attr, default=np.nan):
    try:
        return getattr(obj, attr)
    except Exception:
        return default


def summarize_yolo_metrics(split_name, metrics):
    """
    Extracts robust YOLO detection metrics from Ultralytics DetMetrics.
    """
    row = {
        "split": split_name,
        "mP": np.nan,
        "mR": np.nan,
        "mAP50": np.nan,
        "mAP50_95": np.nan,
    }

    try:
        row["mP"] = safe_float(metrics.box.mp)
        row["mR"] = safe_float(metrics.box.mr)
        row["mAP50"] = safe_float(metrics.box.map50)
        row["mAP50_95"] = safe_float(metrics.box.map)
    except Exception:
        try:
            mp, mr, map50, map5095 = metrics.mean_results()
            row["mP"] = safe_float(mp)
            row["mR"] = safe_float(mr)
            row["mAP50"] = safe_float(map50)
            row["mAP50_95"] = safe_float(map5095)
        except Exception as e:
            print(f"WARNING: Could not extract YOLO metrics for {split_name}: {e}")

    row["F1_from_mP_mR"] = (
        2 * row["mP"] * row["mR"] / (row["mP"] + row["mR"])
        if np.isfinite(row["mP"]) and np.isfinite(row["mR"]) and (row["mP"] + row["mR"]) > 0
        else np.nan
    )

    return row


def extract_per_class_yolo_metrics(split_name, metrics):
    """
    Attempts to extract per-class precision, recall, F1, AP50, and mAP50-95.
    If unavailable in the current Ultralytics version, returns an empty table.
    """
    rows = []

    try:
        names = metrics.names
        if isinstance(names, dict):
            class_names = [names[i] for i in sorted(names.keys())]
        else:
            class_names = list(names)

        box = metrics.box

        p = np.asarray(get_attr_safe(box, "p", []), dtype=float)
        r = np.asarray(get_attr_safe(box, "r", []), dtype=float)
        f1 = np.asarray(get_attr_safe(box, "f1", []), dtype=float)
        ap50 = np.asarray(get_attr_safe(box, "ap50", []), dtype=float)
        maps = np.asarray(get_attr_safe(box, "maps", []), dtype=float)

        for i, cls_name in enumerate(class_names):
            rows.append({
                "split": split_name,
                "class_id": i,
                "class_name": cls_name,
                "precision": safe_float(p[i]) if i < len(p) else np.nan,
                "recall": safe_float(r[i]) if i < len(r) else np.nan,
                "f1": safe_float(f1[i]) if i < len(f1) else np.nan,
                "AP50": safe_float(ap50[i]) if i < len(ap50) else np.nan,
                "mAP50_95": safe_float(maps[i]) if i < len(maps) else np.nan,
            })

    except Exception as e:
        print(f"WARNING: Per-class YOLO metrics unavailable for {split_name}: {e}")

    return pd.DataFrame(rows)


print("\nRunning validation evaluation...")
val_metrics = best_model.val(
    data=str(DATA),
    imgsz=640,
    split="val",
    batch=8,
    device=device_arg,
    verbose=False,
    plots=True,
    save_json=True,
    project=str(WORK),
    name="byol_yolov12s_val_eval",
    exist_ok=True,
)

print("\nRunning test evaluation...")
test_metrics = best_model.val(
    data=str(DATA),
    imgsz=640,
    split="test",
    batch=8,
    device=device_arg,
    verbose=False,
    plots=True,
    save_json=True,
    project=str(WORK),
    name="byol_yolov12s_test_eval",
    exist_ok=True,
)

summary_rows = [
    summarize_yolo_metrics("validation", val_metrics),
    summarize_yolo_metrics("test", test_metrics),
]

df_yolo_summary = pd.DataFrame(summary_rows)

print("=" * 70)
print("YOLO Detection Metrics Summary")
print("=" * 70)
display(df_yolo_summary)

summary_csv = RESULTS_DIR / "official_yolo_val_test_summary.csv"
df_yolo_summary.to_csv(summary_csv, index=False)
print("Saved summary CSV:", summary_csv)

df_val_per_class = extract_per_class_yolo_metrics("validation", val_metrics)
df_test_per_class = extract_per_class_yolo_metrics("test", test_metrics)

df_yolo_per_class = pd.concat(
    [df_val_per_class, df_test_per_class],
    ignore_index=True
)

print("=" * 70)
print("YOLO Per-Class Metrics")
print("=" * 70)
if len(df_yolo_per_class) > 0:
    display(df_yolo_per_class)
    per_class_csv = RESULTS_DIR / "official_yolo_per_class_metrics.csv"
    df_yolo_per_class.to_csv(per_class_csv, index=False)
    print("Saved per-class CSV:", per_class_csv)
else:
    print("Per-class metrics were not available from this Ultralytics API version.")

# Display saved Ultralytics plots if available
for eval_name in ["byol_yolov12s_val_eval", "byol_yolov12s_test_eval"]:
    eval_dir = WORK / eval_name
    print("=" * 70)
    print(f"Saved Ultralytics plots for {eval_name}")
    print("=" * 70)

    candidate_plots = [
        "PR_curve.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
    ]

    for plot_name in candidate_plots:
        p = eval_dir / plot_name
        if p.exists():
            print(plot_name, ":", p)
            display(IPyImage(filename=str(p)))

## Cell 14 — Publication-Ready Training Curves

In [ ]:
# ============================================================
# Cell 14 — Publication-Ready Training Curves
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

results_csv = run_dir / "results.csv"

if not results_csv.exists():
    raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

df_train_log = pd.read_csv(results_csv)
df_train_log.columns = [c.strip() for c in df_train_log.columns]

print("=" * 70)
print("Training Log Columns")
print("=" * 70)
print(df_train_log.columns.tolist())

train_log_out = RESULTS_DIR / "yolov12s_byol_training_log.csv"
df_train_log.to_csv(train_log_out, index=False)
print("Training log copied to:", train_log_out)


def plot_metric_curve(df, x_col, y_cols, title, ylabel, save_name):
    available_cols = [c for c in y_cols if c in df.columns]

    if len(available_cols) == 0:
        print(f"WARNING: No requested columns available for plot: {title}")
        return None

    fig, ax = plt.subplots(figsize=(9, 5), dpi=160)

    for col in available_cols:
        ax.plot(df[x_col], df[col], linewidth=1.8, label=col)

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(fontsize=9)

    plt.tight_layout()

    fig_path = FIG_DIR / save_name
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)
    return fig_path


# Box/class/DFL losses
loss_cols = [
    "train/box_loss",
    "val/box_loss",
    "train/cls_loss",
    "val/cls_loss",
    "train/dfl_loss",
    "val/dfl_loss",
]

plot_metric_curve(
    df=df_train_log,
    x_col="epoch",
    y_cols=loss_cols,
    title="YOLOv12s BYOL Fine-Tuning: Training and Validation Losses",
    ylabel="Loss",
    save_name="yolov12s_byol_training_validation_losses.png",
)

# Core validation detection metrics
metric_cols = [
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

plot_metric_curve(
    df=df_train_log,
    x_col="epoch",
    y_cols=metric_cols,
    title="YOLOv12s BYOL Fine-Tuning: Validation Detection Metrics",
    ylabel="Metric Value",
    save_name="yolov12s_byol_validation_metrics_over_epochs.png",
)

# Learning-rate curves
lr_cols = [c for c in df_train_log.columns if c.startswith("lr/")]

plot_metric_curve(
    df=df_train_log,
    x_col="epoch",
    y_cols=lr_cols,
    title="Learning-Rate Schedule During Fine-Tuning",
    ylabel="Learning Rate",
    save_name="yolov12s_byol_learning_rate_schedule.png",
)

# Best epoch summary based on mAP50-95
best_metric_col = "metrics/mAP50-95(B)"

if best_metric_col in df_train_log.columns:
    best_idx = df_train_log[best_metric_col].idxmax()
    best_row = df_train_log.loc[best_idx]

    print("=" * 70)
    print("Best Epoch Summary")
    print("=" * 70)
    print(f"Best epoch by mAP50-95: {int(best_row['epoch'])}")
    print(f"mAP50-95             : {best_row[best_metric_col]:.6f}")

    best_epoch_summary = best_row.to_frame().T
    best_epoch_csv = RESULTS_DIR / "best_epoch_summary.csv"
    best_epoch_summary.to_csv(best_epoch_csv, index=False)
    print("Saved best epoch summary:", best_epoch_csv)

# Final logged metric bar chart
available_metric_cols = [c for c in metric_cols if c in df_train_log.columns]

if len(available_metric_cols) > 0:
    last = df_train_log.iloc[-1]

    labels = [c.replace("metrics/", "").replace("(B)", "") for c in available_metric_cols]
    vals = [float(last[c]) for c in available_metric_cols]

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=160)
    ax.bar(labels, vals)

    ax.set_title("Final Logged Validation Metrics", fontsize=14, fontweight="bold")
    ax.set_ylabel("Metric Value")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

    plt.tight_layout()

    fig_path = FIG_DIR / "final_logged_validation_metrics_bar.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## Cell 15 — Journal-Level Matched Detection Evaluation

In [ ]:
# ============================================================
# Cell 15 — Journal-Level Matched Detection Evaluation (T4-safe)
# ============================================================

from pathlib import Path
import os
import json
import math
import time
import warnings
import random
import gc

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
    cohen_kappa_score,
    brier_score_loss,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

warnings.filterwarnings("ignore")

# Helps reduce CUDA fragmentation on Kaggle
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
DATA = WORK / "data_byol.yaml"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

if best_pt.exists():
    model_pt = best_pt
elif last_pt.exists():
    model_pt = last_pt
else:
    raise FileNotFoundError("No trained detector weights found.")

# Free memory before loading model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

best_model = YOLO(str(model_pt))

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

with open(DATA, "r") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]
num_classes = len(class_names)
background_id = num_classes
all_label_names = class_names + ["background"]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

EVAL_SPLIT = "test"

# 512 is safer on T4; you can raise to 640 later if you want.
IMG_SIZE = 512
CONF_THRESH = 0.25
NMS_IOU = 0.70
MATCH_IOU = 0.50
BOOTSTRAP_N = 1000
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("=" * 70)
print("Matched Detection Evaluation Configuration")
print("=" * 70)
print("Model weights       :", model_pt)
print("Evaluation split    :", EVAL_SPLIT)
print("Matching IoU        :", MATCH_IOU)
print("Confidence threshold:", CONF_THRESH)
print("Classes             :", class_names)
print("Image size          :", IMG_SIZE)
print("Device              :", device)


def safe_div(num, den):
    return float(num / den) if den != 0 else np.nan


def read_image_size(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    return w, h


def load_yolo_ground_truth(lbl_path, img_w, img_h):
    """
    Returns ground-truth boxes as dictionaries with xyxy pixel coordinates.
    """
    gts = []

    if not lbl_path.exists():
        return gts

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:])
            except Exception:
                continue

            if not (0 <= cls < num_classes):
                continue

            x1 = (xc - bw / 2) * img_w
            y1 = (yc - bh / 2) * img_h
            x2 = (xc + bw / 2) * img_w
            y2 = (yc + bh / 2) * img_h

            x1 = max(0.0, min(float(img_w - 1), x1))
            y1 = max(0.0, min(float(img_h - 1), y1))
            x2 = max(0.0, min(float(img_w - 1), x2))
            y2 = max(0.0, min(float(img_h - 1), y2))

            if x2 <= x1 or y2 <= y1:
                continue

            gts.append({
                "cls": cls,
                "box": np.array([x1, y1, x2, y2], dtype=float),
            })

    return gts


def box_iou_xyxy(box_a, box_b):
    """
    IoU between two xyxy boxes.
    """
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def result_to_predictions(result):
    """
    Converts one Ultralytics result to prediction dictionaries.
    """
    preds = []

    if result.boxes is None or len(result.boxes) == 0:
        return preds

    boxes = result.boxes.xyxy.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    confs = result.boxes.conf.detach().cpu().numpy().astype(float)

    for box, cls, conf in zip(boxes, classes, confs):
        if 0 <= cls < num_classes:
            preds.append({
                "cls": int(cls),
                "box": box.astype(float),
                "conf": float(conf),
            })

    return sorted(preds, key=lambda x: x["conf"], reverse=True)


def match_predictions_to_ground_truth(gts, preds, image_name, iou_threshold=0.50):
    """
    Event-level matching for detection evaluation.
    """
    records = []
    matched_gt = set()

    for pred in preds:
        best_gt_idx = None
        best_iou = 0.0

        for gt_idx, gt in enumerate(gts):
            if gt_idx in matched_gt:
                continue

            iou = box_iou_xyxy(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= iou_threshold:
            matched_gt.add(best_gt_idx)

            true_cls = int(gts[best_gt_idx]["cls"])
            pred_cls = int(pred["cls"])

            records.append({
                "image": image_name,
                "event_type": "matched_prediction",
                "y_true": true_cls,
                "y_pred": pred_cls,
                "confidence": float(pred["conf"]),
                "iou": float(best_iou),
                "is_correct": int(true_cls == pred_cls),
            })
        else:
            records.append({
                "image": image_name,
                "event_type": "false_positive",
                "y_true": background_id,
                "y_pred": int(pred["cls"]),
                "confidence": float(pred["conf"]),
                "iou": float(best_iou),
                "is_correct": 0,
            })

    for gt_idx, gt in enumerate(gts):
        if gt_idx not in matched_gt:
            records.append({
                "image": image_name,
                "event_type": "false_negative",
                "y_true": int(gt["cls"]),
                "y_pred": background_id,
                "confidence": np.nan,
                "iou": np.nan,
                "is_correct": 0,
            })

    return records


def expected_calibration_error(confidences, correctness, n_bins=15):
    """
    Binary ECE for predicted detections:
    confidence = detector confidence score
    correctness = 1 if prediction matched a GT of the same class, else 0

    False negatives are excluded because they have no prediction confidence.
    """
    confidences = np.asarray(confidences, dtype=float)
    correctness = np.asarray(correctness, dtype=float)

    mask = np.isfinite(confidences) & np.isfinite(correctness)
    confidences = confidences[mask]
    correctness = correctness[mask]

    if len(confidences) == 0:
        return np.nan, pd.DataFrame()

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]

        if i == n_bins - 1:
            in_bin = (confidences >= lo) & (confidences <= hi)
        else:
            in_bin = (confidences >= lo) & (confidences < hi)

        prop = np.mean(in_bin)

        if prop > 0:
            bin_conf = float(np.mean(confidences[in_bin]))
            bin_acc = float(np.mean(correctness[in_bin]))
            gap = abs(bin_acc - bin_conf)
            ece += prop * gap

            rows.append({
                "bin_low": lo,
                "bin_high": hi,
                "bin_count": int(np.sum(in_bin)),
                "bin_confidence": bin_conf,
                "bin_accuracy": bin_acc,
                "gap": gap,
            })

    return float(ece), pd.DataFrame(rows)


def bootstrap_ci_metric(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    """
    Generic bootstrap 95% CI.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)

    if n == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)

        try:
            vals.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue

    if len(vals) == 0:
        return np.nan, np.nan

    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def macro_f1_object_only(y_true, y_pred):
    _, _, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )
    return float(f1)


def accuracy_metric(y_true, y_pred):
    return float(accuracy_score(y_true, y_pred))


def calculate_ppv_npv(cm, object_class_ids):
    """
    Computes per-class PPV and NPV from full confusion matrix.
    """
    rows = []
    total = cm.sum()

    for c in object_class_ids:
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        tn = total - tp - fp - fn

        rows.append({
            "class_id": c,
            "class_name": class_names[c],
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "PPV": safe_div(tp, tp + fp),
            "NPV": safe_div(tn, tn + fn),
        })

    return pd.DataFrame(rows)


def compute_binary_roc_pr_eer(confidences, correctness):
    """
    Computes binary ROC-AUC, PR-AUC, and EER for detection correctness.
    This is not multiclass OvR ROC-AUC; it evaluates confidence as a predictor
    of whether a detection is correct.
    """
    confidences = np.asarray(confidences, dtype=float)
    correctness = np.asarray(correctness, dtype=int)

    mask = np.isfinite(confidences) & np.isfinite(correctness)
    confidences = confidences[mask]
    correctness = correctness[mask]

    if len(np.unique(correctness)) < 2:
        return {
            "DetectionCorrectness_ROC_AUC": np.nan,
            "DetectionCorrectness_PR_AUC": np.nan,
            "DetectionCorrectness_EER": np.nan,
            "roc_curve": None,
            "pr_curve": None,
        }

    fpr, tpr, _ = roc_curve(correctness, confidences)
    roc_auc = auc(fpr, tpr)

    precision, recall, _ = precision_recall_curve(correctness, confidences)
    pr_auc = average_precision_score(correctness, confidences)

    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)

    return {
        "DetectionCorrectness_ROC_AUC": float(roc_auc),
        "DetectionCorrectness_PR_AUC": float(pr_auc),
        "DetectionCorrectness_EER": eer,
        "roc_curve": (fpr, tpr),
        "pr_curve": (precision, recall),
    }


# ============================================================
# Run predictions and match with ground truth
# ============================================================

img_dir = SPLIT / EVAL_SPLIT / "images"
lbl_dir = SPLIT / EVAL_SPLIT / "labels"

image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

if len(image_paths) == 0:
    raise RuntimeError(f"No images found in: {img_dir}")

print(f"Running predictions on {len(image_paths)} {EVAL_SPLIT} images...")

all_records = []
image_level_rows = []

# IMPORTANT: one image at a time
for img_idx, img_path in enumerate(image_paths, start=1):
    size = read_image_size(img_path)
    if size is None:
        continue

    img_w, img_h = size

    gts = load_yolo_ground_truth(
        lbl_dir / f"{img_path.stem}.txt",
        img_w=img_w,
        img_h=img_h,
    )

    # Single-image inference keeps VRAM usage low
    with torch.inference_mode():
        result = best_model.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=CONF_THRESH,
            iou=NMS_IOU,
            device=device_arg,
            half=(device == "cuda"),
            max_det=300,
            verbose=False,
        )[0]

    preds = result_to_predictions(result)

    records = match_predictions_to_ground_truth(
        gts=gts,
        preds=preds,
        image_name=img_path.name,
        iou_threshold=MATCH_IOU,
    )

    all_records.extend(records)

    image_level_rows.append({
        "image": img_path.name,
        "num_gt": len(gts),
        "num_pred": len(preds),
        "num_events": len(records),
        "num_correct_predictions": sum(
            r["is_correct"] for r in records if r["event_type"] == "matched_prediction"
        ),
    })

    # Explicit cleanup
    del result, preds, records

    if device == "cuda" and img_idx % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

df_events = pd.DataFrame(all_records)
df_image_level = pd.DataFrame(image_level_rows)

events_csv = RESULTS_DIR / f"matched_detection_events_{EVAL_SPLIT}.csv"
image_level_csv = RESULTS_DIR / f"matched_detection_image_level_{EVAL_SPLIT}.csv"

df_events.to_csv(events_csv, index=False)
df_image_level.to_csv(image_level_csv, index=False)

print("Saved matched detection events:", events_csv)
print("Saved image-level detection summary:", image_level_csv)

if df_events.empty:
    raise RuntimeError("No detection events were generated. Check labels and predictions.")

# ============================================================
# Classification-style event metrics
# ============================================================

y_true = df_events["y_true"].astype(int).to_numpy()
y_pred = df_events["y_pred"].astype(int).to_numpy()

labels_all = list(range(num_classes + 1))
labels_object = list(range(num_classes))

cm = confusion_matrix(y_true, y_pred, labels=labels_all)

acc = accuracy_score(y_true, y_pred)

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels_object,
    average="macro",
    zero_division=0,
)

p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels_object,
    average="weighted",
    zero_division=0,
)

kappa = cohen_kappa_score(y_true, y_pred, labels=labels_all)

acc_low, acc_high = bootstrap_ci_metric(
    y_true, y_pred, accuracy_metric, n_boot=BOOTSTRAP_N, seed=SEED
)

macro_f1_low, macro_f1_high = bootstrap_ci_metric(
    y_true, y_pred, macro_f1_object_only, n_boot=BOOTSTRAP_N, seed=SEED
)

df_ppv_npv = calculate_ppv_npv(cm, labels_object)
mean_ppv = float(np.nanmean(df_ppv_npv["PPV"])) if len(df_ppv_npv) else np.nan
mean_npv = float(np.nanmean(df_ppv_npv["NPV"])) if len(df_ppv_npv) else np.nan

report_dict = classification_report(
    y_true,
    y_pred,
    labels=labels_all,
    target_names=all_label_names,
    output_dict=True,
    zero_division=0,
)

df_class_report = pd.DataFrame(report_dict).transpose()

# Confidence statistics use predicted detections only, excluding false negatives.
df_predicted_events = df_events[df_events["event_type"] != "false_negative"].copy()
confidences = df_predicted_events["confidence"].astype(float).to_numpy()
correctness = df_predicted_events["is_correct"].astype(int).to_numpy()

if len(confidences) > 0:
    confidence_stats = {
        "mean_confidence": float(np.nanmean(confidences)),
        "median_confidence": float(np.nanmedian(confidences)),
        "std_confidence": float(np.nanstd(confidences)),
        "min_confidence": float(np.nanmin(confidences)),
        "max_confidence": float(np.nanmax(confidences)),
        "Q1_confidence": float(np.nanpercentile(confidences, 25)),
        "Q3_confidence": float(np.nanpercentile(confidences, 75)),
    }
else:
    confidence_stats = {
        "mean_confidence": np.nan,
        "median_confidence": np.nan,
        "std_confidence": np.nan,
        "min_confidence": np.nan,
        "max_confidence": np.nan,
        "Q1_confidence": np.nan,
        "Q3_confidence": np.nan,
    }

ece, df_calibration_bins = expected_calibration_error(confidences, correctness, n_bins=15)

try:
    brier = float(brier_score_loss(correctness, confidences)) if len(np.unique(correctness)) > 1 else np.nan
except Exception:
    brier = np.nan

roc_pr_info = compute_binary_roc_pr_eer(confidences, correctness)

# Multiclass OvR ROC/PR-AUC is not computed because YOLO predictions do not
# expose a full calibrated per-class probability vector for every event here.
roc_auc_ovr_macro = np.nan
pr_auc_macro = np.nan
eer_macro = np.nan

stat_summary = {
    "split": EVAL_SPLIT,
    "matching_iou": MATCH_IOU,
    "confidence_threshold": CONF_THRESH,
    "num_images": int(len(image_paths)),
    "num_events": int(len(df_events)),
    "num_predicted_events": int(len(df_predicted_events)),
    "Accuracy_detection_event": float(acc),
    "Precision_macro_object_only": float(p_macro),
    "Recall_macro_object_only": float(r_macro),
    "F1_macro_object_only": float(f1_macro),
    "Precision_weighted_object_only": float(p_weighted),
    "Recall_weighted_object_only": float(r_weighted),
    "F1_weighted_object_only": float(f1_weighted),
    "Cohens_Kappa": float(kappa),
    "Acc_CI_Low": float(acc_low),
    "Acc_CI_High": float(acc_high),
    "MacroF1_CI_Low": float(macro_f1_low),
    "MacroF1_CI_High": float(macro_f1_high),
    "Mean_PPV": mean_ppv,
    "Mean_NPV": mean_npv,
    "Expected_Calibration_Error_ECE": float(ece) if np.isfinite(ece) else np.nan,
    "Brier_Score_detection_correctness": brier,
    "ROC_AUC_OvR_macro": roc_auc_ovr_macro,
    "PR_AUC_macro": pr_auc_macro,
    "EER_macro": eer_macro,
    "DetectionCorrectness_ROC_AUC": roc_pr_info["DetectionCorrectness_ROC_AUC"],
    "DetectionCorrectness_PR_AUC": roc_pr_info["DetectionCorrectness_PR_AUC"],
    "DetectionCorrectness_EER": roc_pr_info["DetectionCorrectness_EER"],
    **confidence_stats,
}

df_stat_summary = pd.DataFrame([stat_summary])

print("=" * 70)
print("Matched Detection Statistical Summary")
print("=" * 70)
display(df_stat_summary)

stat_summary_csv = RESULTS_DIR / f"matched_detection_statistical_summary_{EVAL_SPLIT}.csv"
df_stat_summary.to_csv(stat_summary_csv, index=False)
print("Saved statistical summary:", stat_summary_csv)

print("=" * 70)
print("Per-Class Classification Report from Matched Detection Events")
print("=" * 70)
display(df_class_report)

class_report_csv = RESULTS_DIR / f"matched_detection_classification_report_{EVAL_SPLIT}.csv"
df_class_report.to_csv(class_report_csv)
print("Saved classification report:", class_report_csv)

print("=" * 70)
print("Per-Class PPV and NPV")
print("=" * 70)
display(df_ppv_npv)

ppv_npv_csv = RESULTS_DIR / f"matched_detection_ppv_npv_{EVAL_SPLIT}.csv"
df_ppv_npv.to_csv(ppv_npv_csv, index=False)
print("Saved PPV/NPV table:", ppv_npv_csv)

if len(df_calibration_bins) > 0:
    calibration_csv = RESULTS_DIR / f"calibration_bins_{EVAL_SPLIT}.csv"
    df_calibration_bins.to_csv(calibration_csv, index=False)
    print("Saved calibration bins:", calibration_csv)

# ============================================================
# Plots
# ============================================================

# Raw confusion matrix
fig, ax = plt.subplots(figsize=(8, 6), dpi=160)
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Matched Detection Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
ax.set_xticks(np.arange(len(all_label_names)))
ax.set_yticks(np.arange(len(all_label_names)))
ax.set_xticklabels(all_label_names, rotation=35, ha="right")
ax.set_yticklabels(all_label_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

fig_path = FIG_DIR / f"matched_detection_confusion_matrix_{EVAL_SPLIT}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)

# Normalized confusion matrix
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(8, 6), dpi=160)
im = ax.imshow(cm_norm, interpolation="nearest", vmin=0, vmax=1)
ax.set_title("Normalized Matched Detection Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")
ax.set_xticks(np.arange(len(all_label_names)))
ax.set_yticks(np.arange(len(all_label_names)))
ax.set_xticklabels(all_label_names, rotation=35, ha="right")
ax.set_yticklabels(all_label_names)

for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

fig_path = FIG_DIR / f"matched_detection_confusion_matrix_normalized_{EVAL_SPLIT}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)

# Confidence distribution
if len(confidences) > 0:
    fig, ax = plt.subplots(figsize=(8, 5), dpi=160)
    ax.hist(confidences, bins=20, edgecolor="black", alpha=0.85)
    ax.set_title("Detection Confidence Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("Confidence Score")
    ax.set_ylabel("Number of Predicted Detections")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()

    fig_path = FIG_DIR / f"confidence_distribution_{EVAL_SPLIT}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

# Reliability/calibration plot
if len(df_calibration_bins) > 0:
    fig, ax = plt.subplots(figsize=(6, 6), dpi=160)

    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    ax.plot(
        df_calibration_bins["bin_confidence"],
        df_calibration_bins["bin_accuracy"],
        marker="o",
        linewidth=1.8,
        label=f"Detector calibration, ECE={ece:.3f}",
    )

    ax.set_title("Reliability Diagram for Detection Confidence", fontsize=14, fontweight="bold")
    ax.set_xlabel("Mean Confidence")
    ax.set_ylabel("Empirical Accuracy")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend()

    plt.tight_layout()

    fig_path = FIG_DIR / f"reliability_diagram_{EVAL_SPLIT}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

# Detection-correctness ROC curve
if roc_pr_info["roc_curve"] is not None:
    fpr, tpr = roc_pr_info["roc_curve"]

    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=160)
    ax.plot(fpr, tpr, linewidth=1.8, label=f"AUC={roc_pr_info['DetectionCorrectness_ROC_AUC']:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_title("ROC Curve for Detection Correctness", fontsize=14, fontweight="bold")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend()
    plt.tight_layout()

    fig_path = FIG_DIR / f"detection_correctness_roc_curve_{EVAL_SPLIT}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

# Detection-correctness PR curve
if roc_pr_info["pr_curve"] is not None:
    precision_curve, recall_curve = roc_pr_info["pr_curve"]

    fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=160)
    ax.plot(recall_curve, precision_curve, linewidth=1.8, label=f"AP={roc_pr_info['DetectionCorrectness_PR_AUC']:.3f}")
    ax.set_title("Precision-Recall Curve for Detection Correctness", fontsize=14, fontweight="bold")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend()
    plt.tight_layout()

    fig_path = FIG_DIR / f"detection_correctness_pr_curve_{EVAL_SPLIT}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)

print("=" * 70)
print("Metric Availability Note")
print("=" * 70)
print(
    "ROC_AUC_OvR_macro, PR_AUC_macro, and EER_macro are set to NaN because "
    "this evaluation path uses YOLO's final box outputs, which do not provide "
    "a full calibrated per-class probability vector for every detection event here. "
    "Detection-correctness ROC/PR/EER are reported instead."
)

## Cell 16 — Computational Cost and Inference Benchmarking

In [ ]:
# ============================================================
# Cell 16 — Computational Cost and Inference Benchmarking
# ============================================================

from pathlib import Path
import os
import time
import json
import gc

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

try:
    import psutil
except Exception:
    psutil = None

BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

if best_pt.exists():
    model_pt = best_pt
elif last_pt.exists():
    model_pt = last_pt
else:
    raise FileNotFoundError("No trained model weights found.")

best_model = YOLO(str(model_pt))

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

IMG_SIZE = 640
BENCHMARK_SPLIT = "test"
MAX_BENCHMARK_IMAGES = 100
WARMUP_IMAGES = 10

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

img_dir = SPLIT / BENCHMARK_SPLIT / "images"
image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

if len(image_paths) == 0:
    raise RuntimeError(f"No images found for benchmarking in: {img_dir}")

benchmark_paths = image_paths[:min(MAX_BENCHMARK_IMAGES, len(image_paths))]
warmup_paths = benchmark_paths[:min(WARMUP_IMAGES, len(benchmark_paths))]

print("=" * 70)
print("Computational Cost Benchmark")
print("=" * 70)
print("Model weights       :", model_pt)
print("Device              :", device)
if device == "cuda":
    print("GPU                 :", torch.cuda.get_device_name(0))
print("Benchmark images    :", len(benchmark_paths))
print("Image size          :", IMG_SIZE)

# -----------------------------
# Model size and parameters
# -----------------------------
model_size_mb = model_pt.stat().st_size / (1024 ** 2) if model_pt.exists() else np.nan

try:
    params_total = int(sum(p.numel() for p in best_model.model.parameters()))
    params_trainable = int(sum(p.numel() for p in best_model.model.parameters() if p.requires_grad))
    params_non_trainable = int(params_total - params_trainable)
except Exception:
    params_total = np.nan
    params_trainable = np.nan
    params_non_trainable = np.nan

# FLOPs / MACs
gflops = np.nan
gmacs = np.nan

try:
    from ultralytics.utils.torch_utils import get_flops
    gflops = float(get_flops(best_model.model, imgsz=IMG_SIZE))
    gmacs = gflops / 2.0 if np.isfinite(gflops) else np.nan
except Exception as e:
    print("WARNING: FLOPs could not be computed safely:", e)

# -----------------------------
# CPU RAM before benchmark
# -----------------------------
cpu_ram_before_mb = np.nan
cpu_ram_after_mb = np.nan

if psutil is not None:
    process = psutil.Process(os.getpid())
    cpu_ram_before_mb = process.memory_info().rss / (1024 ** 2)

# -----------------------------
# Warmup
# -----------------------------
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

for p in warmup_paths:
    _ = best_model.predict(
        source=str(p),
        imgsz=IMG_SIZE,
        conf=0.25,
        device=device_arg,
        verbose=False,
    )

if device == "cuda":
    torch.cuda.synchronize()

# -----------------------------
# Timed inference
# -----------------------------
times_ms = []

start_all = time.perf_counter()

for p in benchmark_paths:
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    _ = best_model.predict(
        source=str(p),
        imgsz=IMG_SIZE,
        conf=0.25,
        device=device_arg,
        verbose=False,
    )

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    times_ms.append((end - start) * 1000.0)

total_elapsed_s = time.perf_counter() - start_all

if psutil is not None:
    cpu_ram_after_mb = process.memory_info().rss / (1024 ** 2)

peak_gpu_memory_mb = np.nan

if device == "cuda":
    peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

times_ms = np.asarray(times_ms, dtype=float)

total_inference_time_ms = float(np.sum(times_ms)) if len(times_ms) else np.nan
per_sample_inference_ms = float(np.mean(times_ms)) if len(times_ms) else np.nan
median_inference_ms = float(np.median(times_ms)) if len(times_ms) else np.nan
std_inference_ms = float(np.std(times_ms)) if len(times_ms) else np.nan
throughput_fps = float(len(times_ms) / total_elapsed_s) if total_elapsed_s > 0 else np.nan

cost_summary = {
    "model_path": str(model_pt),
    "model_size_MB": float(model_size_mb),
    "parameters_total": params_total,
    "parameters_trainable": params_trainable,
    "parameters_non_trainable": params_non_trainable,
    "GFLOPs": float(gflops) if np.isfinite(gflops) else np.nan,
    "GMACs": float(gmacs) if np.isfinite(gmacs) else np.nan,
    "benchmark_split": BENCHMARK_SPLIT,
    "benchmark_images": int(len(benchmark_paths)),
    "imgsz": IMG_SIZE,
    "total_inference_time_ms": total_inference_time_ms,
    "mean_per_sample_inference_time_ms": per_sample_inference_ms,
    "median_per_sample_inference_time_ms": median_inference_ms,
    "std_per_sample_inference_time_ms": std_inference_ms,
    "throughput_images_per_second": throughput_fps,
    "peak_gpu_memory_MB": float(peak_gpu_memory_mb) if np.isfinite(peak_gpu_memory_mb) else np.nan,
    "cpu_ram_before_MB": float(cpu_ram_before_mb) if np.isfinite(cpu_ram_before_mb) else np.nan,
    "cpu_ram_after_MB": float(cpu_ram_after_mb) if np.isfinite(cpu_ram_after_mb) else np.nan,
}

df_cost = pd.DataFrame([cost_summary])

print("=" * 70)
print("Computational Cost Summary")
print("=" * 70)
display(df_cost)

cost_csv = RESULTS_DIR / "computational_cost_summary.csv"
df_cost.to_csv(cost_csv, index=False)

cost_json = RESULTS_DIR / "computational_cost_summary.json"
with open(cost_json, "w") as f:
    json.dump(cost_summary, f, indent=4)

print("Saved cost CSV :", cost_csv)
print("Saved cost JSON:", cost_json)

# Inference-time distribution plot
if len(times_ms) > 0:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5), dpi=160)
    ax.hist(times_ms, bins=20, edgecolor="black", alpha=0.85)
    ax.set_title("Inference Time Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("Per-Image Inference Time (ms)")
    ax.set_ylabel("Frequency")
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    plt.tight_layout()

    fig_path = FIG_DIR / "inference_time_distribution.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

## Cell 17 — Qualitative Detection Visualization on Original Test Images

In [ ]:
# ======================================================================
# Cell 17 — Qualitative Detection Visualization on Original Test Images
# ======================================================================

from pathlib import Path
import random
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

model_pt = best_pt if best_pt.exists() else last_pt

if not model_pt.exists():
    raise FileNotFoundError("No trained model weights found for qualitative visualization.")

best_model = YOLO(str(model_pt))

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

SEED = 42
random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

test_img_dir = SPLIT / "test" / "images"

all_test_imgs = sorted([p for p in test_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

if len(all_test_imgs) == 0:
    raise RuntimeError(f"No test images found in: {test_img_dir}")

n_show = min(8, len(all_test_imgs))
sample_imgs = random.Random(SEED).sample(all_test_imgs, n_show)

preds = best_model.predict(
    source=[str(p) for p in sample_imgs],
    imgsz=640,
    conf=0.25,
    iou=0.70,
    device=device_arg,
    verbose=False,
)

cols = 4
rows = math.ceil(n_show / cols)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 4.8 * rows), dpi=160)
axes = np.array(axes).reshape(-1) if rows * cols > 1 else np.array([axes])

for i, ax in enumerate(axes):
    ax.axis("off")

    if i >= n_show:
        continue

    img_path = sample_imgs[i]
    result = preds[i]

    img_vis = result.plot()
    img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

    num_det = 0 if result.boxes is None else len(result.boxes)

    ax.imshow(img_vis)
    ax.set_title(
        f"{img_path.name[:32]}\nDetections: {num_det}",
        fontsize=9
    )

fig.suptitle(
    "Qualitative Detection Results on Original Test Images",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.94])

fig_path = FIG_DIR / "qualitative_detection_results_test.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved qualitative detection figure:", fig_path)

## Cell 18 — Confidence and IoU Threshold Robustness Analysis

In [ ]:
# ============================================================
# Cell 18 — Confidence and IoU Threshold Robustness Analysis
# ============================================================

from pathlib import Path
import gc
import random
import json
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

SPLIT = WORK / "0_yolo_split"
DATA = WORK / "data_byol.yaml"

FIG_DIR = WORK / "figures"
RESULTS_DIR = WORK / "results"

FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

det_run_name = "byol_yolov12s_aug"
run_dir = WORK / det_run_name

best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

if best_pt.exists():
    model_pt = best_pt
elif last_pt.exists():
    model_pt = last_pt
else:
    raise FileNotFoundError("No trained model weights found.")

# Free memory before loading model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

best_model = YOLO(str(model_pt))

device = "cuda" if torch.cuda.is_available() else "cpu"
device_arg = 0 if device == "cuda" else "cpu"

with open(DATA, "r") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]
num_classes = len(class_names)
background_id = num_classes

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

EVAL_SPLIT = "test"

# Lower size is safer on T4. If needed, you can try 640 later.
IMG_SIZE = 512

# Predict one image at a time from the model
BASE_PRED_CONF = 0.001
NMS_IOU = 0.70

CONF_THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
MATCH_IOU_THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("=" * 70)
print("Robustness Analysis Configuration")
print("=" * 70)
print("Model weights         :", model_pt)
print("Evaluation split      :", EVAL_SPLIT)
print("Base prediction conf  :", BASE_PRED_CONF)
print("Confidence thresholds :", CONF_THRESHOLDS)
print("Matching IoU values   :", MATCH_IOU_THRESHOLDS)
print("Image size            :", IMG_SIZE)
print("Device                :", device)


# ============================================================
# Helper functions
# ============================================================

def read_image_size(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    return w, h


def load_yolo_ground_truth(lbl_path, img_w, img_h):
    gts = []

    if not lbl_path.exists():
        return gts

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:])
            except Exception:
                continue

            if not (0 <= cls < num_classes):
                continue

            x1 = (xc - bw / 2) * img_w
            y1 = (yc - bh / 2) * img_h
            x2 = (xc + bw / 2) * img_w
            y2 = (yc + bh / 2) * img_h

            x1 = max(0.0, min(float(img_w - 1), x1))
            y1 = max(0.0, min(float(img_h - 1), y1))
            x2 = max(0.0, min(float(img_w - 1), x2))
            y2 = max(0.0, min(float(img_h - 1), y2))

            if x2 <= x1 or y2 <= y1:
                continue

            gts.append({
                "cls": cls,
                "box": np.array([x1, y1, x2, y2], dtype=float),
            })

    return gts


def result_to_predictions(result):
    preds = []

    if result.boxes is None or len(result.boxes) == 0:
        return preds

    boxes = result.boxes.xyxy.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    confs = result.boxes.conf.detach().cpu().numpy().astype(float)

    for box, cls, conf in zip(boxes, classes, confs):
        if 0 <= cls < num_classes:
            preds.append({
                "cls": int(cls),
                "box": box.astype(float),
                "conf": float(conf),
            })

    return sorted(preds, key=lambda x: x["conf"], reverse=True)


def box_iou_xyxy(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def evaluate_predictions_at_threshold(gts_by_image, preds_by_image, conf_thresh, match_iou):
    """
    Class-sensitive matched detection evaluation.

    A prediction is counted as TP only if:
    - IoU >= match_iou
    - predicted class == ground-truth class

    Wrong-class matched predictions are counted as FP and the corresponding GT
    is still treated as missed for class-sensitive recall.
    """
    tp = 0
    fp = 0
    fn = 0

    y_true_events = []
    y_pred_events = []

    for image_name in gts_by_image.keys():
        gts = gts_by_image[image_name]
        preds = [
            p for p in preds_by_image.get(image_name, [])
            if p["conf"] >= conf_thresh
        ]

        matched_gt = set()

        for pred in preds:
            best_gt_idx = None
            best_iou = 0.0

            for gt_idx, gt in enumerate(gts):
                if gt_idx in matched_gt:
                    continue

                iou = box_iou_xyxy(pred["box"], gt["box"])

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            if best_gt_idx is not None and best_iou >= match_iou:
                gt_cls = int(gts[best_gt_idx]["cls"])
                pred_cls = int(pred["cls"])

                matched_gt.add(best_gt_idx)

                y_true_events.append(gt_cls)
                y_pred_events.append(pred_cls)

                if gt_cls == pred_cls:
                    tp += 1
                else:
                    fp += 1
                    fn += 1
            else:
                fp += 1
                y_true_events.append(background_id)
                y_pred_events.append(int(pred["cls"]))

        for gt_idx, gt in enumerate(gts):
            if gt_idx not in matched_gt:
                fn += 1
                y_true_events.append(int(gt["cls"]))
                y_pred_events.append(background_id)

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )

    try:
        event_accuracy = accuracy_score(y_true_events, y_pred_events)
    except Exception:
        event_accuracy = np.nan

    try:
        _, _, macro_f1, _ = precision_recall_fscore_support(
            y_true_events,
            y_pred_events,
            labels=list(range(num_classes)),
            average="macro",
            zero_division=0,
        )
    except Exception:
        macro_f1 = np.nan

    return {
        "confidence_threshold": conf_thresh,
        "matching_iou": match_iou,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "precision": float(precision) if np.isfinite(precision) else np.nan,
        "recall": float(recall) if np.isfinite(recall) else np.nan,
        "f1": float(f1) if np.isfinite(f1) else np.nan,
        "event_accuracy": float(event_accuracy) if np.isfinite(event_accuracy) else np.nan,
        "macro_f1_object_only": float(macro_f1) if np.isfinite(macro_f1) else np.nan,
        "num_events": int(len(y_true_events)),
    }


# ============================================================
# Load GT and run predictions one image at a time
# ============================================================

img_dir = SPLIT / EVAL_SPLIT / "images"
lbl_dir = SPLIT / EVAL_SPLIT / "labels"

image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

if len(image_paths) == 0:
    raise RuntimeError(f"No images found in: {img_dir}")

print(f"Running base predictions on {len(image_paths)} images...")

gts_by_image = {}
preds_by_image = {}

for idx, img_path in enumerate(image_paths, start=1):
    size = read_image_size(img_path)
    if size is None:
        continue

    img_w, img_h = size

    gts_by_image[img_path.name] = load_yolo_ground_truth(
        lbl_dir / f"{img_path.stem}.txt",
        img_w=img_w,
        img_h=img_h,
    )

    # One image at a time to avoid OOM
    with torch.inference_mode():
        result = best_model.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=BASE_PRED_CONF,
            iou=NMS_IOU,
            batch=1,
            device=device_arg,
            half=(device == "cuda"),
            max_det=300,
            verbose=False,
        )[0]

    preds_by_image[img_path.name] = result_to_predictions(result)

    # Cleanup
    del result
    if device == "cuda" and idx % 10 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("GT images loaded        :", len(gts_by_image))
print("Prediction images loaded:", len(preds_by_image))

if device == "cuda":
    torch.cuda.empty_cache()
    gc.collect()


# ============================================================
# Threshold sweep
# ============================================================

rows = []

for match_iou in MATCH_IOU_THRESHOLDS:
    for conf_thresh in CONF_THRESHOLDS:
        row = evaluate_predictions_at_threshold(
            gts_by_image=gts_by_image,
            preds_by_image=preds_by_image,
            conf_thresh=conf_thresh,
            match_iou=match_iou,
        )
        rows.append(row)

df_robustness = pd.DataFrame(rows)

print("=" * 70)
print("Confidence and IoU Threshold Robustness Results")
print("=" * 70)
display(df_robustness)

robustness_csv = RESULTS_DIR / f"threshold_robustness_{EVAL_SPLIT}.csv"
df_robustness.to_csv(robustness_csv, index=False)
print("Saved robustness CSV:", robustness_csv)


# ============================================================
# Best operating point by F1
# ============================================================

valid_f1 = df_robustness.dropna(subset=["f1"])

if len(valid_f1) > 0:
    best_row = valid_f1.loc[valid_f1["f1"].idxmax()].to_dict()

    print("=" * 70)
    print("Best Operating Point by Matched-Detection F1")
    print("=" * 70)
    for k, v in best_row.items():
        print(f"{k}: {v}")

    best_json = RESULTS_DIR / f"best_threshold_operating_point_{EVAL_SPLIT}.json"
    with open(best_json, "w") as f:
        json.dump(best_row, f, indent=4)

    print("Saved best operating point JSON:", best_json)
else:
    print("WARNING: No valid F1 values available for threshold selection.")


# ============================================================
# Publication-ready plots
# ============================================================

# F1 vs confidence threshold for each IoU
fig, ax = plt.subplots(figsize=(9, 5), dpi=160)

for match_iou in MATCH_IOU_THRESHOLDS:
    sub = df_robustness[df_robustness["matching_iou"] == match_iou]
    ax.plot(
        sub["confidence_threshold"],
        sub["f1"],
        marker="o",
        linewidth=1.8,
        label=f"IoU ≥ {match_iou:.2f}",
    )

ax.set_title(
    "Robustness of Matched-Detection F1 Across Confidence Thresholds",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Matched-Detection F1")
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(title="Matching Threshold")

plt.tight_layout()

fig_path = FIG_DIR / f"robustness_f1_vs_confidence_{EVAL_SPLIT}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)


# Precision-recall tradeoff for IoU = 0.50
target_iou = 0.50
sub = df_robustness[df_robustness["matching_iou"] == target_iou]

if len(sub) > 0:
    fig, ax = plt.subplots(figsize=(7, 6), dpi=160)

    ax.plot(
        sub["recall"],
        sub["precision"],
        marker="o",
        linewidth=1.8,
    )

    for _, row in sub.iterrows():
        ax.text(
            row["recall"],
            row["precision"],
            f"{row['confidence_threshold']:.2f}",
            fontsize=8,
        )

    ax.set_title(
        "Precision-Recall Tradeoff Across Confidence Thresholds",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.35)

    plt.tight_layout()

    fig_path = FIG_DIR / f"robustness_precision_recall_tradeoff_iou50_{EVAL_SPLIT}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)


# Heatmap: F1 across confidence and IoU
heatmap_df = df_robustness.pivot(
    index="matching_iou",
    columns="confidence_threshold",
    values="f1",
)

fig, ax = plt.subplots(figsize=(10, 5), dpi=160)

im = ax.imshow(
    heatmap_df.values,
    aspect="auto",
    origin="lower",
    vmin=0,
    vmax=1,
)

ax.set_title(
    "Matched-Detection F1 Robustness Heatmap",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Matching IoU Threshold")

ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_yticks(np.arange(len(heatmap_df.index)))

ax.set_xticklabels([f"{c:.2f}" for c in heatmap_df.columns])
ax.set_yticklabels([f"{i:.2f}" for i in heatmap_df.index])

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        val = heatmap_df.values[i, j]
        text = "NaN" if not np.isfinite(val) else f"{val:.2f}"
        ax.text(j, i, text, ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=ax, label="F1")
plt.tight_layout()

fig_path = FIG_DIR / f"robustness_f1_heatmap_{EVAL_SPLIT}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)

print("=" * 70)
print("Robustness Analysis Completed")
print("=" * 70)
print("This table can be reported as threshold-sensitivity analysis in the paper.")

## Cell 19 — Final Paper-Ready Result Table Generator

In [ ]:
# ============================================================
# Cell 19 — Final Paper-Ready Result Table Generator
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
BASE = Path("/kaggle/working")
WORK = BASE / "palm_byol_ssl"

RESULTS_DIR = WORK / "results"
FIG_DIR = WORK / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("Final Paper-Ready Result Table Generator")
print("=" * 70)
print("Results directory:", RESULTS_DIR)


# ============================================================
# Safe loaders
# ============================================================

def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"WARNING: Could not read CSV {path.name}: {e}")
            return pd.DataFrame()
    print(f"WARNING: Missing file: {path.name}")
    return pd.DataFrame()


def safe_read_json(path):
    path = Path(path)
    if path.exists():
        try:
            with open(path, "r") as f:
                return json.load(f)
        except Exception as e:
            print(f"WARNING: Could not read JSON {path.name}: {e}")
            return {}
    print(f"WARNING: Missing file: {path.name}")
    return {}


def safe_get(df, row_filter=None, column=None, default=np.nan):
    """
    Safely extracts a scalar value from a dataframe.
    """
    try:
        if df is None or df.empty:
            return default

        tmp = df.copy()

        if row_filter is not None:
            for key, value in row_filter.items():
                if key in tmp.columns:
                    tmp = tmp[tmp[key] == value]

        if tmp.empty or column not in tmp.columns:
            return default

        return tmp.iloc[0][column]

    except Exception:
        return default


def safe_round(x, digits=4):
    try:
        if pd.isna(x):
            return np.nan
        return round(float(x), digits)
    except Exception:
        return np.nan


# ============================================================
# Load saved results
# ============================================================

df_yolo_summary = safe_read_csv(RESULTS_DIR / "official_yolo_val_test_summary.csv")
df_yolo_per_class = safe_read_csv(RESULTS_DIR / "official_yolo_per_class_metrics.csv")

df_matched_stats = safe_read_csv(RESULTS_DIR / "matched_detection_statistical_summary_test.csv")
df_class_report = safe_read_csv(RESULTS_DIR / "matched_detection_classification_report_test.csv")
df_ppv_npv = safe_read_csv(RESULTS_DIR / "matched_detection_ppv_npv_test.csv")

df_cost = safe_read_csv(RESULTS_DIR / "computational_cost_summary.csv")
df_robustness = safe_read_csv(RESULTS_DIR / "threshold_robustness_test.csv")
df_best_epoch = safe_read_csv(RESULTS_DIR / "best_epoch_summary.csv")

byol_summary = safe_read_json(RESULTS_DIR / "byol_ssl_summary.json")
best_threshold = safe_read_json(RESULTS_DIR / "best_threshold_operating_point_test.json")
train_config = safe_read_json(RESULTS_DIR / "yolov12s_byol_finetune_config.json")


# ============================================================
# Table 1 — Main detection performance
# ============================================================

main_rows = []

for split in ["validation", "test"]:
    row = {
        "Model": "BYOL-YOLOv12s",
        "Split": split,
        "mP": safe_round(safe_get(df_yolo_summary, {"split": split}, "mP")),
        "mR": safe_round(safe_get(df_yolo_summary, {"split": split}, "mR")),
        "mAP@0.50": safe_round(safe_get(df_yolo_summary, {"split": split}, "mAP50")),
        "mAP@0.50:0.95": safe_round(safe_get(df_yolo_summary, {"split": split}, "mAP50_95")),
        "F1_from_mP_mR": safe_round(safe_get(df_yolo_summary, {"split": split}, "F1_from_mP_mR")),
    }
    main_rows.append(row)

df_main_detection_table = pd.DataFrame(main_rows)

print("=" * 70)
print("Table 1 — Main YOLO Detection Performance")
print("=" * 70)
display(df_main_detection_table)

main_csv = RESULTS_DIR / "paper_table_1_main_detection_performance.csv"
main_md = RESULTS_DIR / "paper_table_1_main_detection_performance.md"
main_tex = RESULTS_DIR / "paper_table_1_main_detection_performance.tex"

df_main_detection_table.to_csv(main_csv, index=False)
df_main_detection_table.to_markdown(main_md, index=False)

with open(main_tex, "w") as f:
    f.write(df_main_detection_table.to_latex(index=False, float_format="%.4f"))

print("Saved:", main_csv)
print("Saved:", main_md)
print("Saved:", main_tex)


# ============================================================
# Table 2 — Per-class detection performance
# ============================================================

if not df_yolo_per_class.empty:
    df_per_class_table = df_yolo_per_class.copy()

    keep_cols = [
        "split",
        "class_id",
        "class_name",
        "precision",
        "recall",
        "f1",
        "AP50",
        "mAP50_95",
    ]

    keep_cols = [c for c in keep_cols if c in df_per_class_table.columns]
    df_per_class_table = df_per_class_table[keep_cols]

    for c in ["precision", "recall", "f1", "AP50", "mAP50_95"]:
        if c in df_per_class_table.columns:
            df_per_class_table[c] = df_per_class_table[c].apply(lambda x: safe_round(x, 4))

    print("=" * 70)
    print("Table 2 — Per-Class YOLO Detection Performance")
    print("=" * 70)
    display(df_per_class_table)

    per_class_csv = RESULTS_DIR / "paper_table_2_per_class_detection_performance.csv"
    per_class_md = RESULTS_DIR / "paper_table_2_per_class_detection_performance.md"
    per_class_tex = RESULTS_DIR / "paper_table_2_per_class_detection_performance.tex"

    df_per_class_table.to_csv(per_class_csv, index=False)
    df_per_class_table.to_markdown(per_class_md, index=False)

    with open(per_class_tex, "w") as f:
        f.write(df_per_class_table.to_latex(index=False, float_format="%.4f"))

    print("Saved:", per_class_csv)
    print("Saved:", per_class_md)
    print("Saved:", per_class_tex)
else:
    print("WARNING: Per-class YOLO metrics table is unavailable.")


# ============================================================
# Table 3 — Matched detection-event statistical analysis
# ============================================================

matched_row = {
    "Model": "BYOL-YOLOv12s",
    "Split": "test",
    "Matching IoU": safe_round(safe_get(df_matched_stats, column="matching_iou")),
    "Confidence Threshold": safe_round(safe_get(df_matched_stats, column="confidence_threshold")),
    "Event Accuracy": safe_round(safe_get(df_matched_stats, column="Accuracy_detection_event")),
    "Macro Precision": safe_round(safe_get(df_matched_stats, column="Precision_macro_object_only")),
    "Macro Recall": safe_round(safe_get(df_matched_stats, column="Recall_macro_object_only")),
    "Macro F1": safe_round(safe_get(df_matched_stats, column="F1_macro_object_only")),
    "Weighted F1": safe_round(safe_get(df_matched_stats, column="F1_weighted_object_only")),
    "Cohen's Kappa": safe_round(safe_get(df_matched_stats, column="Cohens_Kappa")),
    "Accuracy CI Low": safe_round(safe_get(df_matched_stats, column="Acc_CI_Low")),
    "Accuracy CI High": safe_round(safe_get(df_matched_stats, column="Acc_CI_High")),
    "Macro F1 CI Low": safe_round(safe_get(df_matched_stats, column="MacroF1_CI_Low")),
    "Macro F1 CI High": safe_round(safe_get(df_matched_stats, column="MacroF1_CI_High")),
    "Mean PPV": safe_round(safe_get(df_matched_stats, column="Mean_PPV")),
    "Mean NPV": safe_round(safe_get(df_matched_stats, column="Mean_NPV")),
}

df_matched_table = pd.DataFrame([matched_row])

print("=" * 70)
print("Table 3 — Matched Detection-Event Statistical Analysis")
print("=" * 70)
display(df_matched_table)

matched_csv = RESULTS_DIR / "paper_table_3_matched_detection_statistics.csv"
matched_md = RESULTS_DIR / "paper_table_3_matched_detection_statistics.md"
matched_tex = RESULTS_DIR / "paper_table_3_matched_detection_statistics.tex"

df_matched_table.to_csv(matched_csv, index=False)
df_matched_table.to_markdown(matched_md, index=False)

with open(matched_tex, "w") as f:
    f.write(df_matched_table.to_latex(index=False, float_format="%.4f"))

print("Saved:", matched_csv)
print("Saved:", matched_md)
print("Saved:", matched_tex)


# ============================================================
# Table 4 — Confidence and calibration analysis
# ============================================================

calibration_row = {
    "Model": "BYOL-YOLOv12s",
    "Split": "test",
    "Mean Confidence": safe_round(safe_get(df_matched_stats, column="mean_confidence")),
    "Median Confidence": safe_round(safe_get(df_matched_stats, column="median_confidence")),
    "Std Confidence": safe_round(safe_get(df_matched_stats, column="std_confidence")),
    "Min Confidence": safe_round(safe_get(df_matched_stats, column="min_confidence")),
    "Max Confidence": safe_round(safe_get(df_matched_stats, column="max_confidence")),
    "Q1 Confidence": safe_round(safe_get(df_matched_stats, column="Q1_confidence")),
    "Q3 Confidence": safe_round(safe_get(df_matched_stats, column="Q3_confidence")),
    "ECE": safe_round(safe_get(df_matched_stats, column="Expected_Calibration_Error_ECE")),
    "Brier Score": safe_round(safe_get(df_matched_stats, column="Brier_Score_detection_correctness")),
    "Detection-Correctness ROC-AUC": safe_round(safe_get(df_matched_stats, column="DetectionCorrectness_ROC_AUC")),
    "Detection-Correctness PR-AUC": safe_round(safe_get(df_matched_stats, column="DetectionCorrectness_PR_AUC")),
    "Detection-Correctness EER": safe_round(safe_get(df_matched_stats, column="DetectionCorrectness_EER")),
}

df_calibration_table = pd.DataFrame([calibration_row])

print("=" * 70)
print("Table 4 — Confidence and Calibration Analysis")
print("=" * 70)
display(df_calibration_table)

calibration_csv = RESULTS_DIR / "paper_table_4_confidence_calibration_analysis.csv"
calibration_md = RESULTS_DIR / "paper_table_4_confidence_calibration_analysis.md"
calibration_tex = RESULTS_DIR / "paper_table_4_confidence_calibration_analysis.tex"

df_calibration_table.to_csv(calibration_csv, index=False)
df_calibration_table.to_markdown(calibration_md, index=False)

with open(calibration_tex, "w") as f:
    f.write(df_calibration_table.to_latex(index=False, float_format="%.4f"))

print("Saved:", calibration_csv)
print("Saved:", calibration_md)
print("Saved:", calibration_tex)


# ============================================================
# Table 5 — Computational cost analysis
# ============================================================

cost_row = {
    "Model": "BYOL-YOLOv12s",
    "Model Size MB": safe_round(safe_get(df_cost, column="model_size_MB"), 3),
    "Total Parameters": safe_get(df_cost, column="parameters_total"),
    "Trainable Parameters": safe_get(df_cost, column="parameters_trainable"),
    "Non-trainable Parameters": safe_get(df_cost, column="parameters_non_trainable"),
    "GFLOPs": safe_round(safe_get(df_cost, column="GFLOPs"), 3),
    "GMACs": safe_round(safe_get(df_cost, column="GMACs"), 3),
    "Total Inference Time ms": safe_round(safe_get(df_cost, column="total_inference_time_ms"), 3),
    "Mean Inference Time ms/Image": safe_round(safe_get(df_cost, column="mean_per_sample_inference_time_ms"), 3),
    "Median Inference Time ms/Image": safe_round(safe_get(df_cost, column="median_per_sample_inference_time_ms"), 3),
    "Throughput Images/s": safe_round(safe_get(df_cost, column="throughput_images_per_second"), 3),
    "Peak GPU Memory MB": safe_round(safe_get(df_cost, column="peak_gpu_memory_MB"), 3),
    "CPU RAM Before MB": safe_round(safe_get(df_cost, column="cpu_ram_before_MB"), 3),
    "CPU RAM After MB": safe_round(safe_get(df_cost, column="cpu_ram_after_MB"), 3),
}

df_cost_table = pd.DataFrame([cost_row])

print("=" * 70)
print("Table 5 — Computational Cost Analysis")
print("=" * 70)
display(df_cost_table)

cost_table_csv = RESULTS_DIR / "paper_table_5_computational_cost_analysis.csv"
cost_table_md = RESULTS_DIR / "paper_table_5_computational_cost_analysis.md"
cost_table_tex = RESULTS_DIR / "paper_table_5_computational_cost_analysis.tex"

df_cost_table.to_csv(cost_table_csv, index=False)
df_cost_table.to_markdown(cost_table_md, index=False)

with open(cost_table_tex, "w") as f:
    f.write(df_cost_table.to_latex(index=False, float_format="%.4f"))

print("Saved:", cost_table_csv)
print("Saved:", cost_table_md)
print("Saved:", cost_table_tex)


# ============================================================
# Table 6 — Best threshold operating point
# ============================================================

if best_threshold:
    df_best_threshold = pd.DataFrame([best_threshold])

    print("=" * 70)
    print("Table 6 — Best Threshold Operating Point")
    print("=" * 70)
    display(df_best_threshold)

    best_threshold_csv = RESULTS_DIR / "paper_table_6_best_threshold_operating_point.csv"
    best_threshold_md = RESULTS_DIR / "paper_table_6_best_threshold_operating_point.md"
    best_threshold_tex = RESULTS_DIR / "paper_table_6_best_threshold_operating_point.tex"

    df_best_threshold.to_csv(best_threshold_csv, index=False)
    df_best_threshold.to_markdown(best_threshold_md, index=False)

    with open(best_threshold_tex, "w") as f:
        f.write(df_best_threshold.to_latex(index=False, float_format="%.4f"))

    print("Saved:", best_threshold_csv)
    print("Saved:", best_threshold_md)
    print("Saved:", best_threshold_tex)
else:
    print("WARNING: Best threshold operating point JSON is unavailable.")


# ============================================================
# Final one-row executive summary
# ============================================================

executive_summary = {
    "Model": "BYOL-YOLOv12s",
    "SSL Images Used": byol_summary.get("ssl_images_used", np.nan),
    "Best SSL Epoch": byol_summary.get("best_epoch", np.nan),
    "Best SSL Loss": safe_round(byol_summary.get("best_epoch_loss", np.nan), 6),
    "Fine-tuning Epochs": train_config.get("epochs", np.nan),
    "Image Size": train_config.get("imgsz", np.nan),
    "Batch Size": train_config.get("batch", np.nan),

    "Validation mAP@0.50": safe_round(safe_get(df_yolo_summary, {"split": "validation"}, "mAP50")),
    "Validation mAP@0.50:0.95": safe_round(safe_get(df_yolo_summary, {"split": "validation"}, "mAP50_95")),
    "Test mAP@0.50": safe_round(safe_get(df_yolo_summary, {"split": "test"}, "mAP50")),
    "Test mAP@0.50:0.95": safe_round(safe_get(df_yolo_summary, {"split": "test"}, "mAP50_95")),

    "Matched Test Macro F1": safe_round(safe_get(df_matched_stats, column="F1_macro_object_only")),
    "Matched Test Cohen Kappa": safe_round(safe_get(df_matched_stats, column="Cohens_Kappa")),
    "ECE": safe_round(safe_get(df_matched_stats, column="Expected_Calibration_Error_ECE")),
    "Brier Score": safe_round(safe_get(df_matched_stats, column="Brier_Score_detection_correctness")),

    "Model Size MB": safe_round(safe_get(df_cost, column="model_size_MB"), 3),
    "Parameters": safe_get(df_cost, column="parameters_total"),
    "GFLOPs": safe_round(safe_get(df_cost, column="GFLOPs"), 3),
    "Mean Inference Time ms/Image": safe_round(safe_get(df_cost, column="mean_per_sample_inference_time_ms"), 3),
    "Throughput Images/s": safe_round(safe_get(df_cost, column="throughput_images_per_second"), 3),
}

df_executive_summary = pd.DataFrame([executive_summary])

print("=" * 70)
print("Final Executive Summary")
print("=" * 70)
display(df_executive_summary)

exec_csv = RESULTS_DIR / "final_executive_summary.csv"
exec_md = RESULTS_DIR / "final_executive_summary.md"
exec_tex = RESULTS_DIR / "final_executive_summary.tex"

df_executive_summary.to_csv(exec_csv, index=False)
df_executive_summary.to_markdown(exec_md, index=False)

with open(exec_tex, "w") as f:
    f.write(df_executive_summary.to_latex(index=False, float_format="%.4f"))

print("Saved:", exec_csv)
print("Saved:", exec_md)
print("Saved:", exec_tex)


# ============================================================
# Methodological note for paper
# ============================================================

method_note = f"""
# Methodological Reporting Note

The proposed detector was initialized using BYOL-style self-supervised pretraining on the training split only. Validation and test images were excluded from SSL pretraining to prevent data leakage. Fine-tuning was performed using the augmented training split, while official validation and test evaluation were performed on the original non-augmented validation and test splits.

Primary object-detection performance should be reported using YOLO detection metrics, including mean precision, mean recall, mAP@0.50, and mAP@0.50:0.95. Additional classification-style statistics were computed at the matched-detection-event level by matching predictions to ground-truth boxes using IoU ≥ 0.50. These statistics include event-level accuracy, macro F1-score, weighted F1-score, Cohen's kappa, confidence intervals, PPV, NPV, calibration error, Brier score, and detection-correctness ROC/PR analysis.

The matched-detection-event metrics are diagnostic and should not be presented as replacements for mAP-based object-detection metrics.
"""

method_note_path = RESULTS_DIR / "methodological_reporting_note.md"

with open(method_note_path, "w") as f:
    f.write(method_note)

print("=" * 70)
print("Methodological Note Saved")
print("=" * 70)
print(method_note_path)


print("=" * 70)
print("Final Result Table Generation Completed")
print("=" * 70)
print("Use the generated .csv, .md, and .tex files for manuscript tables.")